# Deney 01 — Video Analiz & Hot-Moment Tespiti

**Amaç:** Uzun bir kaynak videodan (a) transcript çıkarımı, (b) sahne tespiti, (c) ses enerjisi analizi ve (d) LLM tabanlı "vurucu an" (hot-moment) skorlaması için aday model/sağlayıcıları **kalite + maliyet + hız** üzerinden karşılaştırmak.

**Tek kaynak politikası:** Bu notebook algoritmaları kendi içinde tekrar tanımlamaz — transkripsiyon, sahne tespiti, ses tepe analizi, hot-moment skorlama, önizleme klibi üretimi ve `hot_moments.json` şeması hepsi `local_pipeline/video_analysis.py` içindeki **kanonik** fonksiyonlardan gelir. `00_youtube_automation_master.ipynb` ve `verify_local.py` de aynı fonksiyonları çağırır. Bu notebook'un kendine özgü katkısı **çok-aday karşılaştırma (benchmark) döngüsüdür**; master notebook ise tek varsayılan adayla uçtan uca çalışır.

**Nasıl çalışır:**
- Colab'da Runtime → Change runtime type → **GPU (T4)** seçip baştan sona çalıştırabilirsin.
- Bazı adaylar API anahtarı gerektirir (OpenAI / Anthropic / Google). Anahtarı olmayanlar otomatik atlanır, notebook hata vermez.
- API anahtarlarını Colab'da soldaki 🔑 **Secrets** panelinden ekle: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. Hiçbiri yoksa sadece ücretsiz/lokal adaylar (faster-whisper, PySceneDetect, librosa) çalışır.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **hot-moment incelemesi** -> Rota A (`gemini-3.6-flash` -> `gemini-3.5-flash` -> `gemini-3.5-flash-lite` -> `gemma-4-31b-it`).


## 0. Kurulum

In [ ]:
import os, subprocess, sys
if os.path.isdir('/content'):
    packages = ['yt-dlp==2026.7.4', 'faster-whisper==1.2.1', 'scenedetect-headless==0.7.1', 'librosa==0.11.0', 'openai==2.50.0', 'anthropic==0.120.2', 'google-genai==2.15.0', 'pandas==3.0.5', 'tqdm==4.70.0', 'moviepy==2.2.1', 'imageio-ffmpeg==0.6.0']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    print('Colab bagimliliklari kuruldu.')
else:
    print('Yerel calisma: proje .venv paketleri bir sonraki hucrede otomatik etkinlestirilecek.')

In [ ]:
import os, time, json, subprocess, shutil, sys, site
from pathlib import Path
if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            site.addsitedir(str(venv_site_packages))
            print('Yerel .venv paketleri etkinlestirildi:', venv_site_packages)
            break
import pandas as pd
import imageio_ffmpeg

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RUNTIME_DIR = Path('/content') if IS_COLAB else PROJECT_ROOT / '.local_runtime'
RESULTS_DIR = PROJECT_ROOT / 'results'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FFMPEG_BIN = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
FFPROBE_BIN = shutil.which('ffprobe')
print('Calisma ortami:', 'Colab' if IS_COLAB else 'Yerel Jupyter')
print('FFmpeg:', FFMPEG_BIN)
print('FFprobe:', FFPROBE_BIN or 'bulunamadi; on kontrol FFmpeg donusumunde yapilacak')

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)

print('OpenAI key:', 'var' if OPENAI_API_KEY else 'yok (bu adaylar atlanacak)')
print('Anthropic key:', 'var' if ANTHROPIC_API_KEY else 'yok (bu adaylar atlanacak)')
print('Google key:', 'var' if GOOGLE_API_KEY else 'yok (bu adaylar atlanacak)')

# Paylasilan durum sozlugu (Faz 6) - tum notebooklarda (01-06) ayni degerler kullanilir.
STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []  # her satır: task, candidate, quality_score, cost_usd, latency_sec, notes, status

def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task,
        'candidate': candidate,
        'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5),
        'latency_sec': round(latency_sec, 2),
        'notes': notes,
        'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} quality={quality_score} status={status}")

## `local_pipeline.video_analysis` modülünü yükle (kanonik tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'render_utils', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': '719335b9ce67dfea3d9d5c4055d03f42981a35651571f9c11eee8b4de554f553',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nw-Jdtjn)}<^%Dhwk#Mz*Y4}x|G'
            'b*Yio_WHHox_NJ!51{!)7mThU`0pDtpp4MQ!0D;~tyygMV|l#y24lB{^{U}(uzBKXv-Ez0(~+6DnX=Y*83cfGA9Q_fc$nS7JnXVY'
            'x16nHpylVzr$$Vq4_F**`yedq5AYYB)hGs|MQsd+AO-e(f_aata)F#Z!&*zoqZ`nS?X4D6`H?57>avR#fNsqfz<q&Q9CW}GHWC8T'
            'Xj~ptY$_DV+{lI_UKCag2OZ7CABj`@eD90`Rj#_%&+YV}7XZF+HD#~-3Ejk-Z@NAVt1^SNi5C{DJ>@Xs21)7Kn?~<x*N@FlTQT0J'
            'H1F48>%u5N3|3tfNde<Mj<<bn@nk8eXuBQ(>YftFL9xdPSMUVIFI+{n-9jsL7nin56c?ENh=!(t0XSqMN~03yMdQFyX;h?`qElp_'
            'k_z+2uH&SCX-|0R+`Q)>kVF4e9Ga7F*l{TBIfaJq4j;Pv`LKO8^k~y|N1`Oomu!)M%ky(;OE_0>)imNpHqlC$Qn^i}qI|QA)X`_Y'
            '5$i}rH${9?%i0aKnwN(ky8<~HFXTvZ@E%}cYO9E`JErE4+~$~^Q&P=pyHsSeQ+H{p&!)`NIS8k<Y7B{Kt*dLzu&7TXK|L^GYfo`|'
            'ZAe-Az2OQYJNJ6EsW0%@duQLa?E<83U2xJOLZiBvjvNvd_NvEbzjHMm!%mH>*R9U&4a{#wU;CJ7_SVLh59_d{<$-;+ad^`wG5T&Y'
            'Yc}N;rq|!yE+!+AAX4X}V9zecq+(`&Vpg1yfqVKzl0nD**~1}ePqp{miR6Zv4{-}Rm2XhfDwk+2R=Vu(Z<spJY*(|Gw$mv%#E#e9'
            'jZ;jhb$7V^89Qm0CFUXH+97?2eax;OAxG3MyXgmqoI{pjsL3<i#P>jqV|zx*HUR>w?L|?FnUg3-MHrkJ`BtM@QEfIMr3>v}OrNLv'
            '0A%R2;<z20SsserF^#<y<rp9k6-iXs2W*Em%N<jTRqN}mQ_%83TU3j+sbi^HOiX>d=v614J!O(G-feLDs<AM`fdePPN}y>TGKT`W'
            'x_^iT>(Ch+1C{HulPzPr<yvDkJMSBA3)hrtL3K3WVRe2??O*3|0SVO9B^@7*!+f|GH#F1q)QHYJc-V~)O&+c53v7A<|BUz+N8sKH'
            '8+`A1MRhIs&Nl*aWW*#^2#)x!z?i?QM>P`18;h6GRtm;TmBKO`X<_p*$A*?1UKTEQMv3b@)m&e_(NWq=Si`2(YS?0o8#c^q-p1m('
            'GTIDJ8<qNPy}HFxGpXHB4bWdKS5>rRFtoc0_2iL9PCr{c^VPMO@`|?MT602AR}m&G%AKmu)cGIsd-oO{9HEgUeB@lJpOpX3F6sK9'
            'q5bf23wT{IM7!;5>9=jfezi4gz|BH!zvX!KwRY{pwJ(;+3iMshGTt#Y_uy+WF+d%!;qdNZqk{_CjHzA#f9ikJbMpF#cepT{oT(jd'
            '%Xf_^ddtu+@yMsDpJa<L8Bv)cj^!QQp0peCJwCD5o=;HKI;wIMlNVO;KZ}cCxuYLBbI4L%1%9Dky?TvO42QXB7YWA-Y^ymOA8UIE'
            'ZAVD|wDn}MS~;=e^$%xiR<wBi!-m0U3j*N3+N_KX|AI{h+PLb`4>?JlP{1y8wNH^~{PiAn1=$;VkwLT(91NuSxN|UV<lH}PrE)*='
            '-Z@gCjr*YOYcGhq@yvHCn26E}<JdalN?ZLqk6k`>aQx<vEwu}pocOWFw+Z}YG!HR`)`8x#5-lAJ_`YU3>gf=Moe*y}Tw3nJ!>fnq'
            'ZyQ*=Z0ka*OXGcLG~3`uS58OKmgAE35Ks-1uw3mk2UdpH2VxzaB8qOtZa+c0%f!}xPn)?+M(t@zR4vh=jgKe{mAnFtS3ySvsF(j^'
            'c&%H`91ACXH=(P*IbA(1FzOW_pFF|G6mRN0Kb8uQ{b-ZXVt6lIzK2#P_<$G=yvD8)*l56tXUmw3pVD#yVDGKMoo>ywFD%*FDhTuC'
            '8Qnr}mzaz<1%J=ERow+Ui%pm$dn#%EZY4Uuv^r-~d%@d`_adgDyx>KmgpdcyW_E)Nm*KXA^|y)(=_@j{8;2~0P2J+JaGs6m2HGLU'
            '-(4rsNTNAM^@+}9;!MD)sh~JWxtb%FK%`}a^E+LvK{LjzkKk07Gtspofqbv4T<$C%+V^{_SQIG6y<FB^40ZhleZ4UiU>GV%@UToi'
            'Lz5+0*v1Gh1~rr_?RCMz2Q(=iLCH!I$4aIwzDy>wT~?J96*6JdEas6?;^8>n?$SBMd>3wAd{leheRwnb<>m9Ux3i~jp1%6~`OEVk'
            ')#TXQo2XceKx+fzE57fde8lxGammb>06B9EryMl$X@+iXjX;u&3ap4_tzEs=_WEH}%`FDBQ^2T>Xosa)Qz8ZiAt%>a9+zwH!sTz&'
            'm8RS)I-x)m+~Pnn)kY%aIl5aiXoJU+Y%MpPnZ5)ZReRDEz;G$&(K@`9@Kn5atq{TSM@?u2R!`*uu~^9!ufoL&qgk&mqxRt?6?6p^'
            'jPEDl7<?QraKDrb<8$FvI0&iiYoANMVL1TIQ6B#mB_@B*8k3e_*{F;q!VYeay6VxsqJ(w>Dsa@E2u%^<;yV2sXQSU>PK{j1`~_Lh'
            'Ur`FFd9nI~rk++w$t|4LS`o;dICX|oK1X`H0ZmJ-!%q}(+w!%XP?RoyK*ll$cv+bHXR0k7US-KNX|J|)Ky@Y0^1i3alI_L*Mm<d>'
            'Nyp0EiTUd5Rak}z=o(nKBVLSXTRw&@Hj-Q(DQQakbRBB`VP6qH(s_BnMu;npP-<n>SPlx4yfZmI!YG*d8dX?<rEB~WbA%=!?gFpj'
            'fOQ_VR2rtzCh8^KHNqE!m;%27%Bb#xwoIp~oY{1xOI`L*@pqQBPBTfEZ^SGuia47MIXz%{gPBi^ky~W0a5RNUc3nj%tW&{XjIkc?'
            't7%8J3a41h;fY$?9KBOVAO8OI-Ahr}Z)i<1aDADj%SsJyva|xJ50F-D<5H2c1gnz3f$-P5az$~I<p~$CJ@Pj7Q4ua9H1=RoC&0`<'
            'a1VY9+<l#5-ej@C$`R>~%5cX!m=hPZT!c42N<ju_eIv+iLT4=g&m*j31nAo#{?sNaQw~($A)xV5>u%?xmzaYdOkCn9*xX(n&JjBb'
            'X4zbCZ^f*G;Qtntw44GeP6;9B2WP0!5*<)eQkvb%niBGsRlfu0ri{qBO%PU$DUnhhABg%dt_k+FwZQh*0Bf91!SL|XUcI{ds^M@('
            '-UG6XwjvvYAOYxGfFe^oKAseY<>{LN5rhDvG$kO9XCiWGktzC!webrp+nf^?`zWhn$}#2#tus1Ej8qnvg}K^_3Nw*nqb&*`3sjV9'
            'UGpUpM$}u(F}g!Lai~rvn&%sRg>$v3de{?Uj~l!Ji9#wkbDJ`)Si(aw4Q4H)7=J|}Y@>&R+{7LQvi5*h_(U$gL1SB|!p6zK4I22#'
            'c45{!r_cr$FKM=V+)={DGSs(Du5o{St+QgZT(hH*H7307XV40Eq>lI9A>?#|*tN{H&B+B;`QI0~1CT)ZN`++}ukcm{R+>UA@U<O>'
            'bkVC4=C%VRhHg`!(+iC~p;Jy+Mz2-?Y3<x>1H`xjNtpIPA-5M4B5<&c+XCd>ByETrNdttlic-q!+sDjh3Pjf(>&;iaKGFN$LcDc5'
            'u8&H)Mg7uh(Mh(_`q(axLwAz?Q5cF_U1?9Dz^^Y5*l8x_T}}UQn_{Lm<ybX++N}Gqe`Z*48}tmk1TA;wKW#^gEv@RikK25mEhW49'
            '2bAOD10ARVm~vc%aA@s9lxsjL$800YDGB=js+Fkz6{}GF3l^Mum_&b%;#1b^t`bxaD#X)0KA3-jDpYviMSdNXBa$~p`l!XiH2aU<'
            'E_xyXs=taqMoYOwW@Oj`LJhvCycP4{=5h8=oh!Sw#q(R5ISx&`nZ@{9n*APCv3}<&B%QU6zq%sHL3NJ*#j2Ay*&r>^zFysl?ktXw'
            '-l4Tq7aWjUsSHS`XbgCWvGf|SpmL5zVY=NSOn0`qg3S^GnPP3V2-rjcl({T?L4r;4C<>DrkVh*}80gNdJX=I;MhO822_`j>n3i;B'
            'lA!i_6YV(ONa7-#G!EO_=mr;JGl@_oEbh&dz>yYkI_=Od4r&HvMVzXm&j98A$PE)l9o}RiIsJDeznF6fmxzKt?aI5jK_{Kjen(aZ'
            'M`Uy%iGspVo!}i;Cf9Kw`sk&YCBiEQmJZe7(OZ--I5atej$6|zf=%9u<;V8R4E1E2C1@KtFYb^}+bXX+nK1l$d-+&@nJr{qe1^f^'
            'Up5nBs6eTc7UKp|{y*-AZ+!'
        ),
    },
    'render_utils': {
        'sha256': '0b64530651d15439402d8bfc20e22507acc64e8e32650a97cfbd2bd8c0a665fc',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK*_PWzmf$<TBAh<mAT5x&Xt9ba+9xfgY>j0}M=9IwPUTP_2_|8J01Hc|sBE6oe=sloIrBCB1M>@Wmxw?_0AQwMRo7Hcsmc-v'
            'MBLbJ+_-z}?d`n|%RHFGp?LaqwGJ;t7ACVW>#w7A7)MDc7Gb;&vs~nhAPZ+=vJnRl#$0Hu3Js$5CYgxt)gqcMI#Gh}auFmV{7V^K'
            '2631azIdF*!K5ES#WmJW3NcOB8<EaMv50aJrx($*2bE^xGR&g+W*n!}AReqY9fm8S)jG|JoQtp0SqYt<MOip4;*FT68GHx@Rg!}c'
            '`6^w8q6qV%H|TU;q);=tfDvV(DA#cs%yI$M*5xFQ@&)yHLq!JSSs~H{z-A#9VxaRVrZO{el|=xwW4<n_!9c*97^W{T*XuZf`AW-T'
            'T^3>;6pKM;Z*Q+N&(f6`kLP7kX5n}&M+IP$v<QkQP4Z4hzFP$OB912N=kIx%sOKzHPx+!OqFDbbC+jSohIy{u6bl>!42oNx1w~jy'
            't5CJ8e)|Id{*Wdim%|Q2+p5?r_}S^4zBzvL^Xd4P<L5u0oQjdS?g;p^7rqA!f!awC#CyKjn}wHQoUZZ3cpV4RaFNC^JNPLJFQf1Z'
            '@COhB&}q+S2*Db9eTl!tUX<kJd>&0BtPb4^@Z~&!S*FiWa8^!nqQ*F|4S?7}&wl>y&9gVpPsYFg@a*)}$?Ne?FQ1$|r#^Z`lx#d-'
            'c=-y}1|L@8EGk#{6bIQw*uT8nhsBtr9sqiL{OZlKmoLU|emME*Wc=#&%cswt8)Nlm;XEkg!W+sL^b-1nSTF{}&7ouP@zAfn&Qlmj'
            '60Aaa?>z~B4}K}dX^`ZeTPBY_V41Ivj1K_}^)eh!(>Tq>L7pRwfBxYB{yc?0CWuLpH<Ww&)GfCP-jA=+Y=%YdIq#>i$?^T&1M5R}'
            '5hdeGcz<+cegGT~vnezSU!DeW9va``uqc2+<m+GxxCaGf#V!BiUBG;d)A<<C6L2oSX+DqI=JTf@n}%^*=Hm2m>x3Ts%_sEe;m1zs'
            'o2=YDp92$QkpV(YvuI5l1LkkFMjY+x5>)C*5U$chyn*(EY!fEFc)bY1WRnHvyI-O_%r?3C>ZeT}WKjS+#s>I9TIQiy@K>O<VV35;'
            'c#;JdK+FnQU0?hNeSYiZ;yF-Hc&x#BmSMLD6`=hf-aup6lMpuO1`t4(?zu~hcv8cK)5rbS;YA6R2SqWjqs1aPy8i%c9Nay)cQ2Sc'
            'xPNr?aB^_}{=Mm+g8S3M@WI{L{OC`!d*S_u!Pkcm?|&U09UeVAI1GaM*AE`f=Ci|t2b0<Spa#0kVuZUWigiBR-_L@p!9`Ro$_dO<'
            'HiZ&E)CSXZwSSSO7jd|cmGb>n5GDKRJcb83f_%Dvi_yf<G)w?V%}IFq^!cE8Uo;i_jGb#}WdcVa&}g8_aX?!Pj}UARHctp}6f_B>'
            'f4d<nsA>!!0p0GSBs^b=AHy{qx6GcrheLWAAt$n01A@Z=yPu_TJPD>tk<wloh~u(ISJ38EeE;%|$lyRnCL&3%Bs`eBoLG-l3Va0|'
            'z(rKVVIL1C;;T#wB_w4N!32)GK_e*~!v_Bl#-Dt`Jm%^xwD7^hzdbGd`g7C5!@<!my1+Bi{{=}HtjG;CVSaD+@b2vHz1iXX!Sw5U'
            'hlhW<8{P}1^LrB@3iChRzk3+o4<39yc@P9dH)aP1!Pm3N*GIv<@L+!T-sh$Xj1K!eh=02O-POgS_^10%{<gG0cIuPKfHkPrmHD{M'
            '-`J-uv4`Iu12gyXDR6bKU%ogwmE7HrC%^sr^7Rws?z+`hWQl~i5;&+9bi~m+5I5*h!`Fp6^YF)&JoRLoUWCUP!@$q8>L{e2bPD5B'
            'CQo025y6~i^AK6oPOsDH0H+CTexAm-Ljyk8p%1IDi`+A|IOE?#krx?^!3%&=0VrC%_r)0h(%-zE=zjy^(zF;d%Z-vB0SBRF7DCgj'
            'D2Z|;bFwi_t4JQuq>`FMu}L8b3N;8(_X?VWM%`Xw74Bhg0H33Ew>OB>t1#>KsCqOvAbg=lGt|gneFs@dM2To-)`u2vkp;lcAWu<-'
            'Cs~$eHcNzm=6l~oAhE7OSWP{6IkVTg%`dX;BeB;|c@<txQNp9y0@)8L<sdSJvu<<udRrSxPydpuJ@ZlQgtG`HQa2A6qLzT?lX;>L'
            'xbAAaNt%I-1B8AymPjDi!bAyL2+y8;`~zp)bU~|^Kq-5gy325bWSzup^$ZJe!@hzwGYZ#xXK-`)=LNEVo%*~07}+^zY3uV<lPi2N'
            '_hcW2*AmLM;wSC-YXho;K3Ie#<>?<wpkspR$sGV<WMUrM&K>}gRV}DlikmbF$v!w`N+tH@!~*bxU*i(s6K?bt{5w<7&*k)QC9#ob'
            'Ae|uNc^SuDB(p>}0Ec*%X3+)o9_B>qCI(zbqR|zcAiy_;3DrabHvzf>yw)N>LS9WYqS16wCQHPN2#8NN4px&{Fq9<+$USu*+`oVK'
            'LC+TxKv#Q_xEq6Vjp~aoLF$=HAipod_x#jV1Qm`ERJ&O8bd1MGmmgtnV4->bfRYA)F5_90)rsd%!4fH_rjY<b1eZY+6VFGQp?G6x'
            'vtX3DSVu_$JM8p_<33E<phLjE<3mpHSq5_mJm<4S<iOeh8%fYfb%rXhkWQiSJuq%?B%zWdh{gAGLKCc~co33CN<_JsrB?}Rxlj$2'
            'OyN~hqy*qe28U=Lxm@@pb!9YG;C7kv&E#y-Cse+m60Dq|MhsRqL|t2eeMHqByv$+cr=(ni7T`RI=JPN^6(7#aK+PHbt7dE@%o#t^'
            'xO+~e6c+=<<cGmUSacoUPM;ij9;mlNgP!t!<XVzvAR@z@S&mCvy~@)PICW$z@(t|J72yTygb<cS1?Ukf!@3o2*xv`MCHxH=0LPz?'
            'h;@Y(gQInvF6k#%m;qWw32FQQWCea-4}=N)`*k=)0>T@unvXM!Z#{R<QUm-T%4y2FtX-{dbEB+27Q|@ewtd6I;yGhNaweIMAw6f-'
            'bkh!r@L0g_YQ|xlzeDrmEKLhF>0|EoxHphQ444R>Y55P;RBR)5j3=xDv|r6%GpBm4X3&Ik;}n{OZ#Rdmw;iAk{K%^An`>7cf(|d-'
            'IxgNPyE$hc836Nmb&6NRwaIQ{pJg52!eOg3HPOVcM3Y|C4@L+)Zxv{b7(^SJI3B5<25ANlBYC)X^k>+`@E!PccZ+$0Yt@_WK+(yH'
            'Ehw^$J#sqP>{bTwlg~P9L!w}H&f4^}EJpY4(*X$Rx&{%z;Wex-4#=u7o@|OR@2c{A5E_#t7p#tf8cv0NU+h3Xn>~JACfH$Ski95@'
            'S;E=jV@)1caGYFZC7jbi9*IeS3c)=3;MrZPMw5l4`tLR(bK@M}dY(-g)|l#}gBzxrIN49c(%Y|2_<o8SrW5)HsXO#Z!G>@3Zmlyo'
            'n7ecD=iiQBKO6F#$59?6xmdzZ!|7Hlf|d)AFAQ*7X922Br1*oeLT|mD4EDF7ueCk`x>d_cCt2<i27BpCGypEUz0M7Y)*wv130DEl'
            'Kna}ShbW5!Sc+g4iSD(%&1i+rw>_8*oP|MyXtwt>$d++L-~y3{IiRb!2&cdqZGv?aBiw-_iz#7u+Ud0Jw8rCd*IsKF`rI={ic*H!'
            'WJ`FH0jG}^QtYf+a-&hT<e&Ex=7+7Lc$m-8C_J73xGW0wx|8+*ThB$eDgv_B99C%(;*Z|I`~?EsModCtwKT()!YIXm{`-IMRIh@I'
            'FiK^MY<<la(V7&s@+Ea+<8y5W?nB1~A}+Lm9B4=+q{Gs%U`9?=kCMCqw%qC$Aj33Wt*ANd+(0}g3vMn}B?vlY9x7%8D&SfKu-MWd'
            'J%^IHAZ0PIy}*DV&o17{*80)Ap2aQ0Ay%%?5KE|T_knHjT9+b^HG57vvZ9)9l<oOIrtSc9OrJpF0_n=GiWE+*W%xdnThr`A9Tt7F'
            'QLSaEm28!RSj8L$u+ACo7?kYlLL`jxgv!FIk@cKciP$xi48zqVoS~fvS1badr@{b+d65O{1;SIa*##Ot(7N}0^k;<i4WQuTGy$qt'
            '@G9kTKxTeeENJE`$Yw&lLavVcpA*19{0cNTy#gVMWF=NP3)FxoqWgG=<9Yn{?Xahp%+R(l6Q~YA@9kSWO7<&h1<sc)sXxRw;*grg'
            'UuTEs$S`_CL+K@f(&vM7aYq2Kp$Grco1@{mxjOhxF2$sbVxsp&1cK<;n6)lSkwgc4*eSDOF+{P@7mF~uSQPTdBrdaY9$l;gdI3Ga'
            'v(;HbMF5%?XPyNY=l~$PunlAOK-e#U62`c$lMH1~SQgf_5y%v}i@S5W2!MLvmO(cMfU|=2IXf6$p0fZO2sJM8>-!&+g9Ql4AxhC_'
            'RAF|`LMIQ_gVPcX2GcBEd!+ut0cJp{5i2e-q(h!9rJ~9S#cf;=vvy4evJbKO9(5EDDGo>1G{4*7HP7vK2uL>#$7vQ_03C~gsa+HM'
            'j)Zy?B~w)Ag#Mlb5Sy)mQls2+v}I<u8^(Cc-ml{bxT&+r#W>HV=Vx>N_mSH(Rp75aTK@vaG9rW=U02Atg&s6?Yu?nY{?X13%|h-F'
            'RBjw7&cZojXdG-t-Cz4xz5SzM_lN!h{%u=~n&x@FH>niOQn$hwdX4{lk!?kH5-x!}gvn4CXvaI!Ud-NJ&)V!kIYV-9?DR#D8>7X~'
            'LorFy7zE66x`b^w20j-~otb>y$S+qaH>kst1uQDClSn4eD0m6X3Tn#bJ2;@JP-1`hRO+#c-XpyLybR7oSB*wY<7h2tNZf1bt3^tu'
            'DNq5}2ldhSSh_enBv`tmJtiw>0Z@|jfk6O}%V?NGcMk!6AVePS=-QotEd$n2-MPQTmaKpqGmgj)PYu|*yEpKrxtjbCsO_I1KQjx@'
            '@d1{haa&*oT@e2w;*HjHU?B4m`kv+J{53il9NZuBR=ch+X*&#x0x)_6-++3<;6`^34$iOu?5BEz@`LJ80Dl1~0rH=Zj)t?6M>>Kd'
            '8g;(I8SEUG`Z4*&MD*zhpc~6!>{WO2KWgGXVbZPXqj~?MgV8p_GLfvF!J5kxKgcffAqtnp8T<y>`70k;+Y4GP+x((9khDM%Qg&yY'
            'hf}$u#lJB#Kcatle{1K%!DB!H;F-eND}GV!`w}+2-0`#>`y0_;0OkN`OA;W6{TQ{`mtibIB$~kMV5bM-HB!^ulA<b`11hyqr(dFr'
            'YF+?g%|y3=nV1I9haR#SNt*OOgjtHZ*g~q_e_a3}{_f?QAH+K~3-3f1NK!h<!wjfBLmy0m;1(d~6R-BJJ|*v<kIcke0JBKUF>&I+'
            '-&`f)UB3bjvmOR3<oHo_z-%Y%msvRL1DrIO<^4SE&jVoKbBUO1GA4@PB0%|thPq4;uH<}FM+V{zvP=Lv0KqXSQ5IUko`N|+QtC^Y'
            's36A|5D-@|HAA%&AQ%-eyNmP+<Q@=hB<zSNAo;r%I4<H<p#wApOnD{y6($!jQ_wi~Z4%DW8-~{<j@5j_Y(fP%ZcxpMA2mgn<_@41'
            '>@3Sd5sVww5QL`-G{}LZ2Z)hv)JQ@&ww44Rsuv+Nx(F`Aob4lMeaU4%YP_9V5Nw687!VDoroOcJrQz+LN8+F&-P0ACP0vj6$m^5O'
            'sfpV3TVXf;AOwepuMoq}JC!~touVXB?Iu8E`&fVmeTDF2UIJ-pKp|jc`^%7`2ayb>c;uDEy#EjyrN;A-C&d(7AB+tTm?k1}qITn8'
            'c#}G3?kO={DF;C=gUu$c9dM+zp?ADh&-y_14~FNaE=v+*EF}#ve`t=)zI%E<VgdAFA0AS@B*krz+bcqg4=oeX>()_$ySzpXrM+mc'
            'NNL%zmJG&=no6_OvsWj+Bw4lhmK>-|NjI(}FlOZ(-MvPKJ?Wz+Ng~<=Vc`-o2$Dq-_bPjc4<$Kv$Ztdxg-!SIMuH1h>tch=6ez(<'
            'nGTQ?P!ed%3hIPOT3#$j(vDzz;#S9rCIT{p*5VYECQ|bIQqBv}L?F5#$)#5VX;x#RK8D^H2Nz<(c}LpO=3-2vGL@1w<+2HA5Ueo-'
            'p-YP<aZsvqrE^59f^>+zP13)BVEo<llY@gptC4k(=dxEc%^6uarT_il4Lx<yb0{oE+-Km$$s5Qt402WISCAD_Q)3MWRU{Oe5hX$o'
            '0jAFx@FE0e33~vo0h^U|71pC&9(V6JnvY?@&^a-j0oKqt?rMG|n7X=GesHwB%MkEl?2Aj3dZ2BJ4pHttYZwr*E?sZr)mnn8%CD#E'
            '5vv1~Wn9uG)jwCc*PwbvcakV6!<+33s|e(&TyJOUW~2JP65HCFtKNL6&QV1ETzD_`k3I7YoJScP-6SVk(7qJ!jG?{5t;RiMUI6Dp'
            '0-lBW8qYX{hPv-Zbci~@U<z}<^eo#@xf~7XWfB!qNg86$r3WcI;u9I+2exkob9F1&P(<x~HEH|ea13-B{~H|GlcrEha!Mqic4uT('
            'KkWH}e@kfi0y{PG^2FOIbglqma1oB?o{UYpMv%k7!F;=aP0;9B<cV0Ovlw<+?Es<EYb3YHRS>BSC0F@7VC$>BU7f&GoPFZZ!KBwt'
            'R}9F?4t2LiJ`B44h6|m%Ut|1B6Raf=OlMcsy01@Q=`4{Tad!L9RADfU0iBF`ioX1w;tI<?-&ZRtPl}O6Fj4o2w7_^pc32>lmG52)'
            'v+z-jg2N*;vj>gHW+2x`Vp|={D%9GIW|BwQA@+3X)UQT`hIe(^D|t;R(pfafyM|(2N^M$$?znHV1_kX7aG<}F^#t<JR|=Z^HOKS2'
            'cqjhH>B|>%esDY*LTZq!a6KLS#5{=ejgT>R#BsAz(0~5>f5?!xoHVv9k_G4trQPPyOrne=yzqS#r)Sz?0$nWQ8M6DforQ4=gdP-?'
            '(m6FmK1LV-s3QMdjg2{dyO2;>IDm_ZOExkNDg@ibfS(ynAlrCp^75WF{nZOz0s5WJ@gI!lBV3U7g7XSFZ(!_i!vc0Jo$zjUzGa8X'
            '*-=~6W!aCcJt?;W4r)Mwm0eXB2B9#WXYDcylqpJKMqN?SjPl=s+}aQ4bF_^SjtJSwO@sAKuo38;+-^SMy)+WCY)vKx6dBba%Vs3#'
            'pkqGk5vz;##1Ptn6G7<dA&aCGZyVYso*F7jF*%5@f=x~;h`iEB;3g`h?5>Q+1dt+Ck!40P_2nCZNJYAUoVFj}$UwYA)u)#D;kit7'
            '=m|O6b(566(w|7FW#}>ch=6fStdfv1yV%FL$2gJ8acNyGtp-K{ub~rhL>`d9+pS<N^o$URv3)SEJ+&Q%;_TW(6r7HW&6=IRKzK{p'
            '0e(gn>?kSzO?u?9{@-}F^~GQ?P+P^5n+j0VQz9ALxVi9_ioHQ<fdR%S@2@ko|A%alg+6Ixuv~lU#6GvHC@bPa7G8w!`LIO?fkn9r'
            'k}AZ9jd_S8&`uS-s)LdjLEfiv?&A}wk@q{=*_ga~_tgNHv6jjYKx|-9BlL2WG1tIGyUGlvdU^xhK=k<F)im!>MV@&cW&wLg1gmJ1'
            'X16DardaL@<WUQu`%(p+lM75vj5W(mE09?*_7~3#DWHj;e*8rSThGZgh&)e;+cf|x8?azYuprli1a?TgUIde{$RDBTtBkwv-f6P>'
            '?wv@{R)3|qDkr*&W2MCLi_>R<i0B$9gy@pJ`h9@MzL>5LkM7@NEg^1j7H17$EDMs9Jc~khTVTZ(?OIftV-Pi>6gnFDXgIy36Oy?y'
            'lDiFf*w89Ka{>Tb05%s3Ko~?c5u<91w;dw>K+UwCSisbu-#qO<d}P3r#|K$n5R-#BD%GYznX}+Q8zN(COA>}4pd6n*e)bIdkbx+e'
            '5B^FgnC3~11BeB6DQqis4Ef?y)WjeUgKWC!W);<Yn}5}PJG;{xR)2TD9ljmluWx$>TZSNM5*y*2GP(Q`stw`4v;X-vIsfY0U$Fd;'
            'e>kAnFn4=U$7E%yfvW6B)?k$alRV9hPMhz|9?18%<Ii2vkmj3U6Nz+LZisjoEc)Sn#SX|NWA>BuG_vPL_e`r--Z(eiZ~y$o+gT4*'
            '>H5gOzwN!(Pu*|v1I&y8Z-6$@b@#w<#Un}%K+(5OHzLa&_IfO{sJ2O%!Fkcu<q^<a|LErWasrg9>Bdu!{xy$0Y5UeZoUw?*+qOb='
            'ylv0fCL9u`yIVu53O+FAS&j;Tq%1;sP<n}xQ-kLvH>FIwF{J)DlLW9Qz7lsI92|%{G`kWyzI}l2ST!S?>dDS)XSa%#kWYRaA7FN2'
            'yR~>w7BFgD=MK@UD1pMCaaagyAT>|=3Twbgf+gR<ae>5G@f25RU(!(_es#duu48<{*4@Vjn)8CWo2O%Lox(PfSr4ISq57%pMcQ$T'
            '2&BVjdyQr^rsseKWC>GBSv7{BK*!DkecsH$^t3HV6A5HViI)ukVe;JZbd#(y9K=YL$hj^-YmOv9I2IqZ&GZGB%8QFIv*4S8UZap#'
            ')6HnFwN?RJF0fxsTf64Brs*ZMUqIUJOliBR!={X@XNQ#99ow{Vqcmjp&t6H1&MtR&OO*C2XI0Y)Ousa}!J0L6uxT5u!XQ}=?gPaL'
            '-nV?ljXyYOP-m1RVdNo3HUaCAM8!t&!pim;NUs^3SM=L<B@>di0^SF>LgCDl9xs9L7?#H+p$YV<1~OctI|+Wm9?8Oh+1?Bor6|rB'
            'Az=Sa<C0tw+4WNi$GrEh0_N9;H*CM26sSlfVeHo-RBY%)@nMSmHPa3ja&hwm&xtw`<n{3+mI1p`PQ*=zQ5-2cP^x{awx@HH(=G)}'
            'B~Fxe(@2^s!zyo+{Z`RLN~siz3jL<UFFa*x%Ao+gwIvmve-5p~d^%fczM*NZUV9Q>d1_^J)>e{8I?*W-i1tOt0DR_=hc^4CduSoD'
            '19$30LH;I&x}h+QhhCvYu!|APlt4jzBMv&I+|}5&VHl>*Do*j*PI`jAMa;wSl<O=y{D}{I=aUV=2nwC18S29f7{>NX@@Nu=#y6g0'
            'CBN38Z+rDvFS)uE5V@g-a+H3xXNGE7Whj?LnGu`yDaK5F!-;HBE=dNRX+3j?f$NC!Ma6`0FU}mC!gBJtbE_zb9qrL%)C$&?c6U*0'
            'soRS-EO&e1`tsMyI`hLi%yY@E!3=y}EU07xL;()Esr000=ceRk$PYQGQAm<kZ(;N0T;Je#Dcnvn*5`)g^~s=^dp_^nxUlHYZFRdx'
            'bmGQU#(GvCdlr4MKvu|j<qM>O*4MLxa~=eX6FtA>vy?7+bquI*i@G-VuCKQ5uNOwB{0&&xy|Tg`S^PpWy0G4J4|)xue7C}KOOP>z'
            'nE9%(5KHUZuPYG>%J4@mB<%7SFj`@W1gclt$~&)F`MPfacZj3*4fAoY6{_yLUIde4qCu+{ott;I22Ac6MJ4^aoErKr>B8{rvT8E='
            'mi9To4KTVD*rM)bk9N-`?jCurk@-AV0dRF1!4J145}hwa|MUF;Xs1dUco8O{9C7y+i7XZGv<fh<OXo}NZ03RF^z)1H>ysBxPF{;O'
            '?CKDBy3gBfHVP8@j`NJ21Aq%zt(8dJ=u%xt=CGb!;IRFg9Bl~nS%{f@X5=0MT<|KLsd7Fs=v>$o`}3-#uIz)nu1dynlq+os*<ZU<'
            'sZVe4v4`^g;laUyjV(ZED!st);AplTUYkk*;}t=n-~Kf>I6SzE$vjLq_$eUd7<Kf3%H%?xch!URA3{d+oFx>>NTott?j6yLR%Ntk'
            'QUN$a{X>ueUGb2S@IBONdeSN(x^u#pK8OhU8hQI-1A_8EoT5jyj4MhuLV^?I+ACwjbZ^)heO1Oqv?eX~DnxfRlnnrTQC5KgYEc+}'
            'R5~^~+$hdX=q|gVMDXE^dws6_a!kiRa)QLXL7kocHC7Olq#-<+c0KH?)=lxu4Y>>a;~Xug#O2dVV$LwR683LM9ohDF_k@y-$q=al'
            'xvm=n0-$4q;Ns4x+5z0Odjf*1$zqnt(&vB1G5PkLVHvx?m@2Djm5@5N?=&wfx2`7*9TPgnpz?Bs`442#1IyP-<<Fs^x=vou2{ps`'
            'y;uKCy}!fU1{D!^#GyN>X8SwKl1rF^`c3P~v!yIRn~?*3-<*~PuQG(qf`K`w&<|@dzar@M>Wv6zES1JL-B5*Yp77WyhV#idm{(tO'
            '4WhZ+F62?Oc$@9g)cdC0(%gMXW)ZZclN`tnCrcM!+C)*SJ1ZLj+l4I}6V2ZH2oaGoOkiy@%ED99QGxv&9U5E@0Z_vtJ@D+kp{KV#'
            '&-bnyPd<RHQK7em<Mtb|qaaO8Z@bsh2$f+H*iu4yfL59Y8UBy8y58GF&NNl()y-MT;X!3(orDw+tK(Cv;3Fv%uue|tXq?P###uF9'
            'V?<aPR%g}?QCmWuMPm#wi%Ex}0jqq-agbb;!G-)_g>~EnJxZbiQw(szStHq;K0Dv&Dab|Bs99&{9GpaZ`g&FQ<cy<>1Y<eic%-Ag'
            'GWF8F5ib)uU8Q4$=t>>5Q2EBE8y*b`6(yzsrSLHk7Sln`3FW49gwlLkOG0O}GKcFExo@b)VB_T?KyS%hdTPoMdpYYlW)<rVtRBTN'
            'R7!BzKGH!XQ+j{7spLq<9Hegq9b10E=$XJXIaV_%1Id5opFbfIRM}AoCt$tIuiZMrU_zS9Xc~^Z>AIx+f~$3jn*qd~kq2|~(6f$Q'
            'Vm!u})yo%S(p`?YsX?_L>~IF83QfoXTZ?k6X-rW0lrDpr40UXXbwECr+>FPR3K{yImN_lpNGdH9#*fs4FD4)-(`ZNcZD=-%kGl46'
            'RP&PGbW`3sH5W)d4fbc?(DH$G3grW0dkjs5Iiu$;vsP#HT3tWvJvK>mrcX=E8jW7RfSI$h9)fWrU)?~T$YNo&*Qfo9<-ObK+}x}v'
            'cJCL_hrmrmBf69dr1!|)o?7$0EMr*jC=RB<a$t<JMd?Te>6R=Q?OKBhjakn*{zye<<2Gn%p{Aj*8?P%uT<r&9@1f<)z1!;5*favD'
            'vEyBvaLSaEl0V0vXI_`{m7_;W5{XT^>=~@HWO`8LJ`xWs@MB44VN+3SXVEYM_OPmP&P)pKoeU$#x(B{yXz{eJO7@JPD@0t%NH}U5'
            '=Dud%jV?G`yp+YP8LPH^1U##5L~s5scKxvDd~4z8>~L5dqhDoFtQaw~hq}tSyRG?T@fx>^*UZ1nHc7CGrYd`4-n|X;asvmUa*uv8'
            'gL7Lp=T>0;04sD`hKNm;EYwx!npk6kaCpZ?a>6BCVL9bGEiLP-Fq%kN5<>=sTs-G0$)&4WpWZK`i$#noSGv~ZrPV+HY!YLGv?B(2'
            'Eg7(;o@9?CkeZTHjAo);0&H{Ne<Rl9za>L*UZdH@E2&B1MRr&aCncc#RF_+o`jNgf&(qU%&;E$+EH~^g%^t}r4Z0yySE51l;PAjX'
            'Z_HVyllEXiDy<JIM?2P_B@?nm9-SFXOm9Q!xo!E|-59sVSVtTp^9?*#RXaW3e1&t@)JLrQ=Mh6{Ze40wsOxtGoZ=7b>G*K=^wKs$'
            '(d*4>U)l4!)l&0&;&Jb`tg@-_lh)|~Q>7FeUEdh7@foKY`4?Z@YXH6N{TFh}^7#KGZrSZ-gQicO+Neg7C%402OZdha?o=h9ZM8|@'
            '{(~$!9xbZ~`0Oyo3#N9n=(F;JIDP#_+{m3v30S2yY*ndPv(?I&G|aw@H}x}S)Nt%{nwSQIcsn%O{@FyCYs<?6R++hjJinzKNGexh'
            '5hKSW<Qv8r+cd<6ncj^iqB?B07S-%U4RtM}k*a_e9bY(|tyefy<tv*Jb#gnrmQ6}45yGR#!L<pLUU}wRa;mqkW>xJuRB?T9Dv+|?'
            'P387~{`dd#jF;E=u4}&8;+vlN<}Ka#Ahc$X{AG0-a@v|KpVW<1^0nlL+mFebLF?|pE<d~@^ZqFJ&m;?j<@h34yRLl??pH>#s-u;u'
            '?0X!S($z`Mf-GA3BF)HkmOWwD3pf<302;;KP6Iq|6bs2t#keCfN+!-)bVCu{Vi`g0I#+RX0019G-jed%*8<iVFg7ct6lT#1GZLbu'
            'n*pH#g2YW#=)AYvS%Ay3CR+66O9|}k;JngiTVP<u9SEve4MzX)T)}rju_b>A-*R*jC@>`!nk&qT(gK*N(0nY6n%HjF8z82Bfq#9A'
            'e+}`kN1l`9mP0njgeUs?6pJW=s4QFTNjp*ZkV%k6>Rz33!PEv=(^@HXi^2L2tNMiK&Gq$Cd?QWa_Z7e7ZieRixCY-SGy0*bYwd{U'
            'ip?|Ma@U8%Q*$2cTpDkH%5K5a*<kgm_D9EgSvL+g?T#G{Nn*$)EEAHvJ&FV(AgWFDi%(!oG8G3KO!Ys0uc*`E!NY@HJio57(KY^b'
            'gkB9Q`>z(Ro3B)?|IycADS%%eA%W!kr*xvrqn*8KAjpk`cLxaf(k(rX#+$bqXtQ+{msPuVA6Na4zE+Eq{wc?*T~+Msw#?I@!HxS@'
            'L=AC#dWs&!C~9()0)-e*>^S2VCih*ZOQll4l+0mFSzidyNq|8$2mt+}!+<h{hJ1)vhXuLnaDphdxbWq(l&H1LL^_w8EGL9Q7$icl'
            '2X=6oMl-Iqr#7!Vvw-YwD&0vq+Y{k@9!@#FJ)KvYnq`m$bjfUR!C7Q4SOp^<2kXaW(WfZcRZ3BOGK7$NVChPDb788MHnzZE1xEVG'
            'K)q2#6ixSJ2J8lV87V2fCz5H$6^mef75DxDIQK-%g4G2w@SAk0BD)rxqsLTqdS|Ds%6*p1({u9zFMf}HaYJ!=Z*XvM$M`z;UcnxJ'
            '9p?Ws6y)Gj5sBWmsoZaFxs6p0S_|TE_8m-3m>nmx(?u{#uZDnG)>W&&X2JRtJ#&WQ$b4@;pZ#+0j-W!hHJ+!WOdpCD0p6I4x2+`l'
            'iS<#hvIt1Z<}u-)FHVUAHD0_d3q(-$vy#%h??6UTufI!U5SO0;yB$q^@iVf)SPS66dA$q)2t49w|3&`$8~n@r(-+4WP-=swDWUP`'
            '9zX?}A1JTHJ|FKY1ZwzPJ-k*Azxd)r+PJNSk@I*=i2W<H;$Lg}y!C0ZG}Z0F3!8Sq3zryfea`^T6cPXU|NO((59MS3&_AI6fm}fU'
            'UGoub)2ZI@VvA+``|#h91r`n-_&>XuN%9Y!J8VUU6QX2|i10kvpoDVDT8|S{a>!|B6xlVQL{NP3Ce)YJ@#!rIr{!|NgaPHD_dfry'
            'b^g53dv=m2S=>0M8!EY1d`RaOo>HS~3&$GGxfwKLh(8na)&!O<-S3Mr4Q_0=LKV(5m^S7Z2-Do02)D`nqub2S9B}9Dta<&Wuon1M'
            '7^tGfZZX42w+{46@q(NJ0HH;3mD<t~9BmY5GElm4B?STQU&2w|01b2RbME4`$mu|fH;;@vn|{ZTJ0n{a_Li}*0T{^T>7_hu=K(T2'
            '>geoqR*C7fT$M4i;CToqJYLj?7g!BC%^FC|G}VA^Qg&j^q+g%HK7Kg-MBQhj=Cyf=O(kr52OVV{N2!e)RO_bfsy6iPwx%|KI$CS('
            '%rqe8u1m4Wn$P3d<XYLf$~l-z8ZokTB)Y}MnXX2GUjFpZJmL0~C`qH0>zsSn*KglW>*vEizkNG7+-|RRcemb+K%7%Yf@rQ+A7wLE'
            'w2w*m%%LBIx~}5E2GLNXeX8brPof}BFVI*Dr<toGBul}z_8!dzTfgZZOXCoi=c|8Rb-MA=%A5S-B(Hc!4y8vI7@#Js?l9MUf1>1v'
            'G0&GbG$}n=7pQOMH`z+dCBhWMpkE+_+C9RJe1v^TqH!Bc!_5|pkG>gkew#}X=QrJlxM;r$abezqcs4wIaPD03B10fp;oM!M#dwvn'
            'wGZ=zsH+h3Fy;Ur{-|G5Eg2F;H-F$klSQ+z$@cv!!f0N3c?#wxLd9ShE?GsWvgP1*r3~*w>yB;(*w?W`AkJJFP5b;ZNP>8iNBI~*'
            'm>>9#ZnAGs;?s^+mxWh+%Ucz1KV77Gn8?~jMAC|qxM81_M=m`pt&*><_9MB6E(lT)_fKKGFVf;E>fd#iwS>%tbD335>b!EkE>Os@'
            'pIMYIMZeEk&sKa?gM2xS!ysekE~QlRoI=Hy4uye?EH<;RF*g}ZvWU`R8}0{M<to<`v&0}OJIEko_V~&ma^sOvy!Jxi*;K*dN_Gjk'
            ';p<D)RJC5J$}$2=Kru}t7eURgj&(WNE@1pEYROWnudLG^-`HlYpabBB#+_oaGUa^5D^65uRUi?S0NyZAznUDN7_~9^Y7Pu)+0=Ev'
            'K5hc%_R@EUC*htCUE5}q(=|*9&>9+vkHPhYRI~PaH(cI4-vfy{iX&9D1j4(fIoW!^y=<a9h+*qpMDb8u6X-9p?ZCw00j8&ebKpa{'
            '6v&VQoNTaXDNMR*b-pp?Z-@S5zGqJ?8aVlB+Y_}#DS#X9UhCgm%o(<U&QP!2p1An(O~X>`?3oLod+oe|_UkLoeXPB&_f>Va{Rkj^'
            '81V@0WqM6_P41$;6@_SkMyZXj)J2h)qQ@pox<mH+Pw9kv_BahUkT%_T{oAK>py`~mX{FJfMp9#*eO}L4ExYiUUGV&qmTWgja{8bb'
            'O_zK!Zr2V_n<e&M)cJv=nHxwjfd=vxX%c<FOBKZu7`BK`+Cn7+1?n)12EvvSydB7n8zWzaNt8qaQ)-J~6M)D)@C<W*l`Yqqpb_Jb'
            'k=dz@#ZWFgxG6Oi$5=M?(unQWYzRekr&;tn-L<d`Hc7CQiQ?R?0L0T+%dI$*t``Q`>T4R8pPPHFVNf<c-fnm`+EY7RV7RqHEyH%N'
            ')4>i!ALN2}o3P_UfF$s`#)eQ4<51t`9^TQoj554SGRk4*oC|9_XC{?dyn(v*YqVKSZ6$5ln_|}tQuyzu&)M9o?~SgxVbvr7?}!_$'
            'ngc+{C3Ez@z5LNdCC|#_^{E`T*;nJn8~c(b?H8mYySqkCvt|j0-;J44QL2A_Z<eRt8cwQq4LukVsAf^7pY^4CqWNMr<E96?9l*%S'
            'q)esii;<d1*+e^*?H|^No0IlEc2Lz5GMA=`>Sn*HzAofiMgMf*n;&oCtgTCbglD#{W4mEfDvl!c>;%c2*p18&s*quiW@}<Y6}oiZ'
            'fvZ0{w&fQ8Z>w}Ql{b`bHv8UCsma!RLlwtDZI`x688+)1VVdaBQusEi0Oz);rljWNHb%D>NR~nQ{_$>OO$ugLefj#?_s?D&KWAq@'
            ';$E~jpqXbY%9VHxBn@wkM8C4Wq6^inquQi9lWH~6LF5DRjMd<?G-Rbcr9TG>`TY4$C<4EEbBd$Fac2W^$ai~Ax7FcAvt_|ILHIKL'
            'bXZ-&!h?u%!@&{joB3>YtN{D%^M&p*>0HOuM^b~_6@9-F!<V_9*|$#W@N1uK+cf7T<M1VUXOew?l1?^4N|y19GLB!#&t1v!ko*5Y'
            'dHgemsH0a(@|uQnB3PZ^Mim?AOm{BT?Y7cmzeekkApfO|vT!U(rQxlwr7yQHJB6&D7%Sm0jaCs|*qm{)bQj82iA<UmhJ{{I{DIwT'
            'se`Z?W-2Uo0ct>#jRJA4xN0hOK2&>!GM=Z(i{G7$GtYSAop0^&s}9O5S)4|9%!$Krk-Q#KDAsaxZNc8wg>!U40M>DnF5MfwVXDJR'
            '+(B3~4l<l#x*4tObgj2_FZXliQqQ@ra%OewykqPpx+z}foU7jSoDzf868`S41N3cCF(IVI8>dU-ZXe!E=*E8$lKe&K<RiKX!23P|'
            '3KRqNo5AiW(HVM^W=j9_i7|?w%w<VxZZn3>D&tnps-`vGr7t!&Asy0Vw9&bKnr7x(Q%T$71mpEy8~QCO&?E;dlDjTEbpg~ZYT<pc'
            'Du`0okcKsj+dx;9GA@kyvp#pNU%owf8yu`Vsm(`?4Nw|4{O;+L%R9I*wXsis{_f4QH_uPTzy9#-^wr7h@lP+GoIF3h^#=Fwwt2d='
            'mhZLI2bMBVQ<Ib`y1g@BWZOrMs6F?pt8mp$Z|Q4q>(#FF{7Z@5nvH8br8Jz@-{Ue8Kd3{02Ean<8?H#e-2qi`lS4DN8e@&qbhB2y'
            'LPA|$<5pvb^!)*sEyJw0FJr^n8Sb%;vv8CvT2OB4CS{PWHO3bNskO$r8FL{DJvP+_gE{aFlt;Rod1pcY!};|Qy3p^K@80lsbFc%%'
            'ZDzq!)~9y0RNKbbt9OSHL|lfOKt#((0RKIYBQb#!Mr{^20PvEbQlu~;@Xb8~yJ?n$%OS>A;gM0m`S1wk`Xoq3*R>X5Te)XpwSJPI'
            'V9X%nr9)+0w5-b4@UmmAgO;KUI#s<xBd6WanWw*c=MA7_6gawT)kLp$aQIdEIcRvM5N7nwmCYKVx`o`nR<fD_*%sg$;o3!9s9KzJ'
            'W}tfC1XF*l%|MlCtPYRgx51;Y4e(ertbvoSo1yW?8nDV~WyI$CMwG0K`aU}^_Z%>`0be`sGy-#c0DSUlwohF`C(LAs8zoHeLS6d='
            'rxM+!BiU32iG4wsnR}HU*izVZtjk(SM1eMa^>dod^)uaVkMs%J$O4SzgPvo&&%xk+1Eu;1fH|p6#Nk8uPv&)yLr@ngU|_@%M`D=D'
            'D)E&Ct0(sN#Sy$bvUp{EtwJ4&{L~JnQ+dA|=3;w-_2<EAea|y_ZcO3W^^`OsC4ps_N3p?XH{PA?o374%x;2)WLd87y%sXyurqt^f'
            'W^u(|n?5T2Rj@`{elWZo9?*3;`b982_}26~?N5hv*;pJ+-XA@<M}>J&@ewn4OH_c88Qn5=#1Plf`|*55UpD3Cy`zJ*XWqhd3R@sd'
            'hawR72Zu^)k%fJEH8tuOj*lP1Y=!Gw<k5TbxKBdNm;?gZ^OvV5rv@}sX~e+>FLYJ6%~Ur>lW!pcW7x=nl6D5~jc1JrT`XsSBZZIm'
            'Gwi#yyb4>kt{T~Up!>7u4a6}VuvtNxo>Ph!Ku>=I9)+F>yoz)lunUXGY5ZDK#93#h!d3U2LKwJ70~+N`2Z`+}k-8NgSB46jD45{g'
            'TvLpnH7>{T`oWae##!0`Uk1TcB8-jcO1>~OXpGo?!}Zi78`-LjqjS>a;m0PXak|6qPC@P{-pMYx=}JY?uDr^eV?bVqo>bjy$O~=R'
            '#p`YEzVS-!cxm)n!Uzgf(E(B{BMxnNB$R2aPw(Sm6MTqJa4AvQ7v;e!_%Kjp+s<q=61SEq9d60g1Zc%2qQFaKr^}IZO>|tf?3W}}'
            'RyenlAgmrIT}$24vBQZf^OqAY-LCbokhdXbdWWo6)zg8~?JT_CX-Rw0#FsjHJkr%-io>1w;m)>NeavqNCw&p25v;l!@=Nircx9id'
            'rDZm36$YeHMJFvjT<~hLh;+sRrXZXySS718&-&_XydaxTu^cZz>Gy#SBTTxmj{UnZ8we9JzVSm+HYkb&xVlGs6(rFd#^KBBx{yjL'
            '`o*jUXJDpmG2}~zy6imI02TDjtG*;WoL8JOa#5p^A<53go}*WO_59~k(oh>Z_&sWwsOBPw=Mo}8g0bM08K@VyM1?ie#v~o{JdKqw'
            'iMg0mIs|!LVPPX^$F?;ES;8EjdAU&??y$j3rK>uU?{}qKPoNshG%52{APTySSpZtZh#QaOavVQ@^V{*iK0AH(uM%?_90*PUAufYV'
            '07-#4{{ZU~O~VX!H_#!`U6miA<$kctfhS2|T^0#2DB4%4SA%xU$SQ?Mb=(!m(d0oQq;mQMgxE9-v$x(X&||=60o@&d2nJ4UOj0}@'
            'Sh?UTCd!?&p0x-$W#nvR&MNZ^6yb*LWTbur6pLK)jlUdQOqn;0N517(5y^&tMs?9|ckW&5dRx8Wb2u!s2>r8cV!G{lJAxx6ovGv4'
            'jykLvQ44_!-`Blh#6hg`)YVstof%(b!Fr){jOf>#cZfcA4STb1G48(2fxBylPU5@QOdTNu6jIqBW}hj_Y1G_$mZ6pp|GJTucD573'
            '-vYZi@VHY;S>rbh&Bll-)^KD>F-D~FW@HH_ey0{oj2eqkUG3plk;bFCCc}oKMB!0Qpf)2u>)HSpiV6d*%lURmSBWudRE^kmYz4)r'
            'LE&MS*Yf$OF4#MnQ!L?o)FQw)X0j?Lys9!x1&<1T;<w22@V@BQF~Lavn357U%3M_yiTf&nqAcSz)ElR5jz&W&14TyObxY4Ql-}0c'
            'qYEl@cg)+>Ya?gsq^QAs?`)m{+Re^!b-e4^)NH+Lx4+vVUgC>$jw;EsOOR>K_v%e>?nrW!P@CS|oU{RTkhFREj^nEIDT`&(e+@=b'
            '5_h^ijKDp`9d}*xblLh=Q0}OTl=rP@+>sG(>e?cV2-?wIRc&<S%V<ucRkeS&AGvsX<jqS6=)}In*aKODer&y4ET2AacP?LbIS{YL'
            'cuLpVzefm<lm*T{jeKWF=SoN-@r<v3Z9Z7I1V!}MbTv{cVp$YKqPh1ZT80>GKg)vO#T|WG7U2?Y?!fYd<auqVx!~BAk^p-hAa{xZ'
            'Hxvt1MGqk5!?Dxr0shs!wJIoTk<{UnJ%Km`>qUe5hgY$t|GBmL$I<aA4G`Vrr1gEh8<jz~P)<Mx)601rT#&SS2USk;QVeRXw)B5k'
            'bs-TaMxVp!Mg~uq3#Qy(X)!wum?WJ?q@n$Bv@8gBl86EUvS8U{bC=Clb<rusxo`2NB*K1uC+y_Q4Cg}U+Mdzjpdp{ri0`y8-9Nt)'
            'N4_|`k6H0LrbW~l%$b@Lnl!#b2!7D4Ch1idA3gvpHz<oKdVHr$O}ee%@TR<yZ{G8yT3b5X%sjT9)~d*hg3je}hnhe5xx0a}w7Z{M'
            'hkz$n6PW5EOJnq+xIk0mBkARQ8ETplB`6w2$uvaC9<$ShOV$CKp(;1^wquD*WYU`jivXn44{@Myi|>3#os#TVX{5qS<5}r-48L$2'
            'WtAw82rIqf)GM?ceuy#~TZ1(^p<#YDm2DJFJVq>K&0Cdg&_zuxK1jTxQfsG5`{txi7vxiC%+pxk{<LEXRZJ!8({R=8Zl95<5H*C%'
            'P^T0V*)YTb=IP2g<Qe6#ba`%^KRkfXgOGGuR;ec=C)u-+pHjwFR5>hry?VAX6W5H8s1^WDkd%=uB7pN4P1a8LHrDQtsX5D4w?2v<'
            '9L74YwfzONYIfXUZ+i;ti?6vf$aJ_u^9V0pkUb;X)Y2!j*MCL})_s~T^R=8cr+<!!{W-_c5|1o*#L&Vo-~Fyg)X?ElyHyAV(*)o#'
            'W+jNfQ#gPohMwuimoI)k{plE;=#_3sQZFPNjyl3qx+cGww1tUH+(fHd%qDCMys^g>HHE{IsT|P1i*!{2O?$Mn8@22qw<fe!cPvG{'
            'Op2|jt~|<D!3=tV9+{*~skm`ty?|tZYzmudMik3V-RwmPoY}bA?!~Q6L;&AK-gRF)8EL)7Z~<Aj@tX^v(I<3h`)m4*@k);89}#EU'
            '7qGFK;YWw<)Lt4F#v0S6$aH<;D!L8d8mPI`i`vP68_#{$+y|mR7xf+}<-3{&uO)88%_Qd;inZ!!YnVCHqK`Kq8b6(qrrHT}ADddJ'
            '#**^#&Y{?$m-0;Bvg&9PREzJdiZ(*sVa`FsxQ(&LhNE4rw1XD)pw(IH7I0X1q)_|vlc31Z_F=0}SQ`Y}IyEkt10o-mCQFsOvn@R-'
            '-uY$zZYit$Ose^7b_%U#mEJ4ns}-J}$2M?sc5e-hsf<1ocYw7@?uIO#cu)(TM;2V|X5<!W16X;V4I7-;c2$!2f0L5et|Y1cE_I`f'
            '$lckr!qhf6DoaBPkeWB!P~1Wl>^1wfnU&kL!d}gD&8*d;*!F7fX>O$k9k=H?q1kn$dcC=dnK0%i?X|8{%Oi86B3tn?=Fvl%o9wcI'
            '5#d7sxQf!edfWbNseho1tqy7|iwdHl=?vC*l2#{T?J%uV%SIwOEki3!58LbPGJI!^QAl3a<20BRCezfQEOsg=jeo6zHU*eo^404q'
            '!o89VT;_zYAH*dg1vVd_=45aQr&_b}6|X^V{NRqquzb-TLl#Dv(a;OKS+(0Xh9)bU(G~Nx+mRNr6ZyK9PY#4uDG?E%?OTmORIUJ('
            'x6RP;VCmTD>y9h$4D_mirb<pC%YGObc7dK~+GhiOELLTXX*efAQMjk^mv~_g3^z^Kxr4THA8(t!*oZv3K*<T&@*L0tfCb_zN@nR5'
            'r<UE2^I24M+zflcbOGXx76zb`=@Ok2D48r?a+ty)m{l2re8b5DvCsiXZgsE{vvNwQX<4Eqzq%JMji5;wf$}nm!wX<&0DVI)?Lh3p'
            '<e8o+Nf^$^(M9<e^~f<L;yY1Zz7balS-X%p(57(^t#b4-lpY~+Ua&Gs)FG-(fP)EU=spsM^Frr}Lgu<Y-V{?gOfj;^&N=ASWnRFS'
            '`pZ`CmmMA~mX}{O(@xg<)(IzT4~%q3!=0M*#(FXl+2I4MY$PSC10l<z`}P(T&QO{+DD#@72}4J(uWqS-^2JfFdBzsyDo~oqCJEBe'
            'S<0)E^sDg%sSk%aS1*G?y>v05pd-yKPjOesU(uA->RK`;Y}7(EaXa0)ol6c)OwI-LH|JktGFN+jnMTtPF<yRYcR#7+GjL(MZ#C!!'
            'H;RUQh#!~upgrM1dpqr!5O!ri0P2TCbR1H)w3+?2y)lPNBlh+ZL={XDP-Q~Dd*8ge(fy95oAws4V|=_-H6HHH*swEGLwifNO5t$3'
            'K>qedZiw*z0T!(vhnJYj4DaWS!#?Z-i7&v77THFuQ|t|FK@4V}Y^dtN19z_sqV*=3(9w5rFh;-ZauQOY*(#9yKRR}EF<^Ax1D7x@'
            'aus1(h3V_0fJVH&FD~#Dr*wS!gycYWK4*itV_k`<?i0>JGU(HBrXA7mz;TFq1uIB*#437^r!mL<bE-ddOlL!)KpiM-4rO48H`?^b'
            'PU=j``QTT#1>1Q5#>!m8k=01paQw6yGCu;R8D<421^%qdNgU<Kd;phGP;%g|sUxLOW_d|*PXalX#9Fs7<b!@*Cd?r`K7N5LfMh@f'
            '>&QnHDMuD7LU;RRm`g`ctsE49-X(ce1Lrmev;wZF%5%WBJ!Dw;+T@y<hro5w{aTuG_!>h!t->(wQrtnW@`6p$NjgK{)m&U*Y$9?Y'
            '%m-~i#ul#t6;i4Q3vCQ$LHSy9c6<%yfPhz&w)ryRAcS{uiUu={#A>AS0T5V5CUkr6e3>g#j)zzN^oQd<98KzO1LlHo9yp^W{G>w^'
            '`LmJ`2R+)L^T#iq0Aq)Eh0EGB1;(9{<yyMt>xBTyYMAKM<*jPmzR;&c^^Uz4)%!jsMimt1DM!LEdX&$C+3yhZ_7TI!cs)h8o0%vz'
            'Q~OR&SJjz!V&8aQQxLx+Zjv`Vh1-Xgy%d8#*5q_8S!kX;c!3f^9)|k%h%3n*!`4G<-HdMGG*e}KDPFGmCRLG6NS4o-%a%6X-oA8T'
            '?G1pfC_*@ZKrp}+LE9zmHo8F~EF?J9GC6ar3o0hzBEYBYk&ISJ@-z+!3~~3k2ba)to*ItalDxu<V@7Tq8Ch6qR<Lg-gH4#5FVZXO'
            '5XeDMk|<94b7TYgyHBjZZbfJu(wwT21B_(fW+n~BbjPaxg=tkc@?<sxR(YW3doENW?~1Bk?`(7Hf#@ci<V=pey<V-iddq(94p)F%'
            '`rOiTbGPmD*SIg+%s1#jtO|KnoJb=WbU%P^grbf2ayKPHY)e*%nNac=CM3#W<IU21LpLK*uEA9>!vqOUZD0=5TILvO|C1NkHm|r{'
            '*2>n~WJj#$Xf&d+jN~)*T#{V9cF$=oI=&fiWx^M?zXNKL^1$r1XT32J-f*s)PfBxR4&#qzyJ;MwllsPJ@2`~sXYdD7;27PLnQ-XZ'
            '#8s4YiiUX<;^j0296A~Ltc>UyhH8oeV4-$l6V>nejVVd_o9U|&n78Uqny=H9ej1hgs_`<I<mz&X>JGh053iKdBj^|irpL+tJ#)gV'
            'IbA8-lw`S<D;oQ3%vDdH1vT)yvkEuhkv9vL5oMa9{Kw`MO@Iwi11bSqP;Sb6Rl<2UR9T{oE>xuD?=Mf#KSJpLMj`Y+R)Mr)8Z80y'
            '{t-d{j|lpIM9_cc2>M$D&)*z3&u2etb<rB8?##}uxOIzftO_&F__?Foj}uH>xkBGOJLb_KBDV}`uP*;?;+*})f$xfF|9cB${pdK>'
            '|B8UpKN{)P66N(*jPUvc(Oovi{MDkm7^&^m;&<j!)oDr1=;0g227iZXqi)RT{js<(-yFk#agf*@@?HKzp<>0s=MNE6pBgm_=DGGc'
            'V#&?|e)D$}OlD=^{M!VT6$k$aCj0+5m`qOnJ}*aK-=tLqnME9BCc51Cs6d7EBaG#E_Tu#Tg?Rbm__t@`$;tQ6o<Di^+PagMG=P{H'
            'B@lSgLIToMngkob8ch6%;pU$`9PNhj{1t=KTn_^>1=X&~yCp}8%qCNzuDU+3SHI+rg^Qn+9m_UfA^RMN=f`hOUTZ_pKs&K$XAq=I'
            'fNr@2hTcgNvI(P5GRf&4yg=Dj$f%NvI881pBN&QuVLXRpHzTVXC8Cexj&jTj3S}2X>mvD(;T3k2;9VNO9OIQ^xavybN<w7g72X!8'
            'np5l2)AlT7$6}LP&6;&%Afa3k!xd+@oue&!u<|Qu0@Y@OxEUH^)mUn{#@Z{;*3{kt(+p#lok9bh1>1TlHG~zGbB~+wzcC=`|ILU>'
            '(^9gd55`rNPWYusjNR{&24gWdfU6=%kBB$|Z@0#rZL<)z7~4{0uAByq-Ay5&H*}`k<llxj-UO={w+jZr)%q)Aj+yvBd-3?>`N>Zw'
            'r_aQ9CqF)a_S?(XZn~u|$Ig{>1k8Ywzy#W9k_9~6V3ToXC)my)G5ey`S%7rCfQFPNswwjX`T)9UVQbNwrr{1=1~d>au`_8h#aKR&'
            'Jn4+u05w^rc^R`Mq6XZ#?&l8#Q8sk!ZxzDV<SD0o%NstmR<~X1TnP5pkBtJf?}GRn#RFE8UpEYU!#0!g-(NgodsyJ-3I+Up;eQ_&'
            '^5<v{(8bl!CG3o5F_Z~Vb%&<Gam#ho4(Oiky!xoXmp>Bh(jMQ^7SnQ@2$lan{^5VX<@=-j%m2Uh51+>97H0WRX&2fTX+K?S7c=eN'
            'mPnT}zMu*hk-{f70?;%q(Seb!dDbR<mgP`9@x?OS_*$<F0v5XMlA&D;1_ND#wbopNwNDr<!3q~&e5F*^-Yr0JCP{)M%My+|rT;Hc'
            '-Yv-'
        ),
    },
    'google_ai': {
        'sha256': '6a2cc8b2916cf3485b8fb85ca1b4b4ad217c00dec1f962cecdc219a2c7bed49e',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}d4JnDmMHqaKLwg~-&m}YWM`>#x6|r+DqBfu;w59tRbB1!P?SX3Y)Pb=r0ghe-_QQe0)PM)SxF`L&D`JAZ>A%Y00<l$oDFAj'
            '9OuxF;wT*Z^QOrBbk_3k{Ui`?<M?JCh;Cmr-Uf>>3dNp)9}C|VNxaH}L`>sEWVc}|mP!1NV4Ssgc7FNkS26ghe<4oJjxLXTqW8D{'
            '#o%J6Rr%k}j{yu3@ED7DvVzYAcR{iiQJe*%IG%~oDx7B`%u*5GM`CyrM1I&loQFY_HQMdAI~0C2*}3(T$vB<_6ETEgf_Z26hGyJ)'
            '(SFsM&i(XucPPSW9!5c13~v2QEc{6T!-{aRjMFq6!I!gGG+=GBD8AphzYWK?g2qZ=dV!cGL7<i?#(pIHakld3^R-9<e|*~%jS+MW'
            'f~9WV%EBPoIe&k$H-P^?e}4pPncM^!yo?uO6ok<YJV_#$^k^+;#Bn%*b@_8K^`jPmxQUAsjj;<4!Z|EJ_&duan1tiZhZ)1`)g+{!'
            ';@)55%5pe{Jnys9o><QPF#_4o4t92gc-9tUm~<AT!WUx#rD&u9Jj+^PGzpeLgwtpW^>e(2=eY1-;zIxDZQKhUo*)DuxCzq?w(UDH'
            'j-zRKvw}(HKoI|2#hDMoP5sq8OJRr?Z7~VcaSVH%0yLNYD4d7TG!=KDFNQp+HiN%Chp`*H(EtL#JD6)mpI?akFuR5E;6DIh@L@R*'
            '$FQleFY!HggI6!xVi}HSxYkJkoe?e4EF3cqizQ4CaUNQJ-=+=3mhd!%Rs@CK#h`!E9rTXmGUfs7Vgl`7wPE2|g0tQeBOkvJPp9x3'
            'p3x}dFd464IwOGgjJo`>E$;kzI3eHz#8!Yr2r8J$a+Qg64ZA<i@FW6abx)5(8Z7)sK7jszYKwV%lgdK^=y(ShlL$Y{g2ggJ=)|-w'
            'ev*Y#7zw)kxh+yZ0^oiORKFle;NXt<?BN-}1B{S`i6naMlO6GFzl}hRQ$m$ASo#Ufyr}^VlZ^ZYbQ{M&7GQf2nv!hjU0v_&oXXP-'
            '$hk^0Jlp`e7972Kut4ndqcwnzXFdX?KuiwSbT*UBqz*t}e(s(3-}GTOx^KJv(~E&Pd-JA$*zX>Tw`XT>f#LwYIyyVNJn5Ydx`Y1N'
            'DdKh@&U@XXlb&$i`RmA^iAj8utmgj0cibA<^!fkow8RM=i~doHkO}z2-)7k|J=og=`gKDfYEOc@JxQnhWw<wq$LSv9JPc7d1VH!#'
            '1il9c4DqtTbPh;hQ^-fD{tHfEhlpbn5TGznuowjsJ&GowwTF(#a{%MO^j5fAu-!cW77=5St`b1C0FQNJ%Qc2KxD>YyAEn6-fgG;^'
            'A@StReIRp4;b1(#MxX}SdKskcZx3bDc!Z?rwZ$}+eu_AH)IS_th%`o|PBVX<N)$p0xg%(qi+DU^f(euZjvXQ0#nC%BJJ4eehr5ly'
            'hry#6P-t%sM`k`m+8iu_=s38Ng+F)1zk~xshW>>U5Uv*V^ESM>bwmTsL5l$C;tp&F74=E@{NdU8dGC;r_bzSI6cy_k0lk?Xzy#sI'
            'BSHZA7&hV0c<C+Gj~O#X+<N$zs10oRD#~~}PtOK${1$M|CXAUle)oNU55UkCXVWPhZQwySE9e;k;{Y-NV@VO<1u>QwsGxrbn32x{'
            'h!C-9N#%jd?Ar_pzLF5hfx%Kd`)hI9gVQW7dZ$MhOwLiDP#<c4Q2dt;@X$k1VB(nv0r7z#%<LNYa0uJQoQVLnn-a$W5~H>o5e0}N'
            '@LnI7>O2jQ%O%0s&%!urZwGt1bI;q~vux1<Di;h9FbLW?5l{FKFl(Wx;YHvlFvA;QI<O<~b!MDf9vFGvQW}I7%s4K;2nPnI$%6-z'
            'RuXVaW^tdkC?XFy!`Aa59QlC1kgk?^ju54gS$cRRhp&lo(!oJS2JlOPGN{?W_{fs7;6t`OWzb(wUE%@!1=ycy_yJZ61Ho3s2?~1h'
            'Gm6vEGy<Pa?;9<6n9^6Yg26%DP)A_78o^fG=0o8OUh7|C83XD7s>4TtrCeknYQ;oCfKGtmFb@}q?l_v`FM%Hh%>E=?o5bbt0h}z5'
            'E7lJ6MIj6ZbZ|l>PNV{NE5p;X2qzJck4(Jx#~~a-k%f!E*&cX!Dq1`(fL;$-J<^mLSRZa22;{wE@4WXG1oHD=hf+Y~BMFD$@a*(W'
            '|Lx^@_ebENN4+=Q%i{qo0wo3#PFHY#65!^*g6Y^Eh}Ci)`;)Y(A&Eaxj$mN{<;f=en*cE(;Uj+tr#y)fis{ZM$nJw6Qs)HdEE1}8'
            'PlAzX%_Ihi3%V1)+B=Tp>=51Zyy*%_kLM{-!H*&w7g2F%M?M?*Y4GZ$`s)_X>pUE(KmU=&k@_8{>Q@3=6)*H(f%?UWuKvR9;fWxH'
            '#lO@vCL|@Pr%h@J$364;=f^mb9hZKFvr+xt!=F1l--s_MjC>=00_6y>86>u~NP!NHVW;SHr2(NY%q$!cO$Cz7#JWXn4@C(;cGwQW'
            'W*BO^is+z2zZ((;;7uA6R*UI8TrLsaV0DOGKFIQ{Sf%kQ0X{Om^`jfsLrA3f2K5HO{uBpHK~N4zu>xY>{=zWW`Q|{J6JnswWn*0T'
            '@bK*Nba1iv{`~CUdni8u#>q2TQej|O^aBSZ>3(qsKtRqR`b3UF0br$V$c)B|Vo&0<M?JknA}9)1aQIdzw*ZlW0h3iE!YK;F2tfoV'
            'aIdT_4iRoRb2`BnNa;Z8Cos=J@dAuOdZg%i1GqH|Huq6M=k3P6uqaS~!s!%FJdib5?9ct9-nomDo`PD;+Eogily^s|_{aj<JKf{s'
            'vtK+3D(^oo&jwxZ{7lN%9q|SkDM2|Wz3FPS2x)IqCEdYVN!OK-hbaNxgAPNoK}e)?vcfY@f||d^0ihdcPt@&p1v&=&br%NrNXVZi'
            'i>U5tT8N`*Zea?K_dwPHR6`Te<>(cldk82C{YJ1`i}uc&^Ip#z^n2&tQ5O~u45WY9gQYh}`*ftxg0y~xG%gK4ikY9mgb#ox0#{DL'
            'Ip7?OnS?VKRsh+`=+PRLv$?PJCWi;iK-GeM0SOG6uA({g6E^|M7Gbgofa{KdYt2^kIdq?sqLtvVfE&=`CjJO&yBg{y08|Zd2w{vG'
            '&sLG_*LV_u1Nib(jd$zM<{=Q$g#fCy_Sa!?$-Z3jB3Oq}+u3pX0Q&c$F^-q(1eshgaTQ%a9X8(ciNB&mLqZ`l7seMUQoaV1pVKzZ'
            'W7d!ntSH51K(Ny>Xx%910qMR+jXVWGfmHKFkoiE1eIj-#>Npag&%3{PC%vP7*BhL9#s)L<%klaG5dHGP`?-63*}H(p4-5y;g5moM'
            've-CH;Y^Tb#!GzQk;&yYp2H0B6(g;e1pi#YR!p2Gj{tH5qW=V&z|O5m^v3QuU3`dUAAT**1{^03&%+vqc!Zf9{?t3^dH}EsK8eqE'
            'c6P=flco}#y8ZV=_<N}GH_m|!!G!4#_kic%IOl$d2lM_GP{tRxC2oP6au>!(Q6_C<zB@axBjRZSCt80Mr$DCkm)^i~QrNxK1U3&z'
            '9e3OV@kpp{;@)RFg#hhxMEV$YKP2T$Na_S?4q6O`!G~ZBbjoA&^@wNzWhJf@#knnFJ}2G+)C3KGG$2~j$h=5PN7MjS9f@kjMmaqp'
            ')C=SSxZ_l8qKCX%{4YVkR&IoDLgQ{4tushg0g%Bk!4p08KhAx%Dq!GnfLT&XLwP#&pm5StJn2A8{aZNqK*^&CP@M#b2S}fLQZ!~c'
            '7Rd{cFF#%QGciL>R~Rk4oul1~iOgul+y<D?+E4u%kfb*7SK=3Ie;$P&$G+(PdfMk^M5`u2aY*83aD;D&rUgWb+Ibrw8ed!zx*+)z'
            'grA@N`6^)H{XT(H2-{6!4B+9IWJtsmX%lOrR=PkF+XN+77zTyLmKJQ;d@F6YUqf=~eL#+N2nTQ^S3p)bR&#-;f_y$FngH@Cl01;Y'
            '=c|+~lJ{_!TVqhf0F%)Mf-G<zf<O$7Lgb}sJYO-A1AZm#or~V#d2ir#kNe#Vlh(=e1w6qy5Dm%h-hoyFq>1fr9`HWzUGG=NmBeig'
            '2(cOkM9Xx?U-==ngPy;hT@EgP>@iC%_6K$LpZu}&v;WX<o9gu#$q1<5ApR@*6Hci8;qlq!(VOG$c~76I{!w*DP-?s^o&`~<<K^Hd'
            ';0o`0r+G*JBkCIIG6=@E9%^*hr~CIyXnyg&*E{^lo}kqnE#z@j?t$vq8Gi%Fx#T<^?7>m*bkOe}dmyO2=^vw{;Yc><?5Ky{Y4@ZD'
            'L%_=<XyY~mw`e5Je_nN4fA?D-_gjCy&VPFC*7d`F^VN&T|K-5pIy^h>{^-g5{;7BTzHYPiiF6nR0cAjMi2~qlvrto)hCEv?+IDV$'
            '9w8wOXF#oh3G99+ehq-W+=L(JCZTd<?_F%vog9c=jaa*)aTlz85v2H$3B0keY}ajyqu?e;4$R?lfiNX?T}d?nnmt$zvKfDp;N*~)'
            '`twP6gIs_o(KbigRVHfz#~fA^?y6=a<{h4Ne+5Y|T7kF&=N;spC7i2u>eG2djBnor8E`;jNCps$`v%Q4#0U1&SGwad=t1-hWll{9'
            'H=U*8zQ_&Gpv;e236@po4k2pHRst6_z!)KjhsFRWeg!@-fDecxl&w(my?dd7^dvCHAO^?t1wbud>Z2B34nasl#mb+b2U9IRbeUkG'
            '#K)EdVB9fq*gK@=iodgxvC5R`LSL$Z$dTU0_mPw*hDoqYKvdWpf>uxeGMQ(+3}pZkH;Krq3lwSlX-ZR0AZ^hs6_C;({{7<YRPzSD'
            'q)Sot8tWA)4l@t@8Q<8Tc||HFO)?q^yyX#iTS>MOy4phfGZJbb{HafBGzdF4w@A3d$WPXaOM$YNi!1cQ^U`^orJjfCiZcV`N7_q#'
            'z(Z*1zk<}m>ntU2PJx=>Pmkb>Gy_LN(a-bX6l9*)NP{`(yEx<l8b_KUAC(@6e~|##6xR4n-m#`7572+|tK7FE>gl1IOIbLFHckOL'
            ';6L_@d_wR#|D&RjnjN&uK>G+9)>GH+AtwrL)qS3q@31G|HsrsBz6?+93je|eBa9fj|1Wd_ts+}%Smzh21Z@(k;VjJ!F_W<sA2y5+'
            'xOF4a+kCKmoAMz6chLHpfXc=vtu4;*Fbt&)o*4w+7~z3|gS7%VnlyFPpW8}SHs)tcEypA}Pu5&Y>vD1kI4FD5IL0+|fJS_WYz@(k'
            'lMkK69vLRb9^NM9-@w1Dm0qaR-PFghmHJaOjImNl_MO8RhzKZY;`B^WAGEvRL9bfvgDvdRsXNGV)y92?z@XkUOx!{fbs!Hc1S~W%'
            '4)sA1|Ern;c1zPf^`J|?@`*5}@@NtPq@a=r=-m_I8#u-O&B8xG1&prvMGLRQ?*LLoC^e?eUo2WrC%E&V;q=HDBA-aeJQh?5{BNoN'
            'B%8{=9{J46hdD4F|ElO2P{%AG+Nc&}E|H0<rgR|9jX!g##BK;wQh)ZOJm)bsc_1(VIyF$VC;BHzoZ>}kJBaSW1W*wr3v8tZ+JXY{'
            'fYu2Eoc&q5BKT~>AETWYRU1NYMdLuY2<FUJi6i1A-NKJnz{+4f0U0Iq5m2-ZnW`E9+B=R5y9bA<;VK;(d0<hJZBJKo5DK#KZG&l8'
            'QJE&itxAVB_^bNtv(~GZAekkgpu)zmuz~8UR#9*L1ojl(`-`U7ZSTVWc)Nl`rQOyp8S56oJT!GuV@D(<qb-nfi2}Ra(uTnp&5h81'
            'r3XZfDi~q@pM@PAB-6By)e_P<&m#qS0jQXgUU)JfnuBN+-k@y*gbuWH@GhcYMEi(0SUfQ)BhmG(w)wL_#aD}nGq8cTeBnjlMwy74'
            '$h#fkPZ99)-i8Jw&y%(os`cZ=|1y=xXCqvyCYk0cuM~zzSCu@RST&l0D%4#ib%0nnkJ*Ht$;*lhz_)y{rgcYy;uBcA#Ac9f>AqyF'
            'UxS*B8|>o5^}y6L%5gyNzJKSVEtkZKlxT7TkI~GfBZtDG&;#R$509=}e?F!TnpC1ium)YHd^`@sgPPGFlE>E|SKf_R89ddqcyxGl'
            'gV%QStglq(Yigq)-9ZztYTk4aG6qzPS4@zIceAvWzq4Sf2Bw`~Hws|(<v8$`8Y>z@0Q1Sj$Ne`ElBr`tLas1m3p=WbAJO_IZUP_&'
            '@K~awKS6tZ63l=;@Pzq<;?|ib7N=HC>8TUV6Q^j*HJ@shtPd>Ops1_l71SX|0ag-3Pw5KPeM#6<4?`&q+D#>evuF$vGbzY7b1M={'
            '@(L|F3Xm41tQyBs3)G+GQEc*BCaa%E9qH;UzCpJ-wVz6zqC0)sknt^udrDa>^JtfD(3M}LXs{AKH1&V<$FceLkSvIFU<6ReOiW@R'
            'eSyo?$NWl)=_ua{r(&q}ZurCyr-;Ig=n4aoh^i54r;JpQSVd`&?akxyELWKE(w}@k$w9@zLo-Elgd=@<{VYGu`u3lE852LfwJBwO'
            '=54DX2`IQnRR{TCOM<1gWtFUeCgEs!HX9>2a6uGANWGXs+TV5`-Tyg`pL6)sQPUF47_`QThrbcq8|p`eI8%}<yORldduXePV(fIo'
            '^#Q%L&}R=X*^@is%C~ft?T__cA9+T!e<x4QyKOQhloU?2RC<o{fX||5hK7L!U*H}z*E9<jk8RCK9_3S7@uRvkV~uiT&7PH-{K0{}'
            'WhLnD8vMAS?WOLn*b}=_3pSoNg>5*6DZufWgzl~+ZKrg`$4a}!tso>S-Ff9RG_>zB@Emh`4@M_L`Enf_f_mByD1*2E5r<Ku*i$Q`'
            'k9r9Tss=D`gZa`!pR=4v*Z@?gqG<irM5|e}r;@FkEym&e1|<V!<P=DZ+RoRwsp1F+_R~k%yHVkb{~EQ<+i-$2V^a)~t<egN?`g^$'
            '&i&#yQs(t?j;SZSg;iI~=$tEes>*y+i}_8tDi19(ak6IiuFt`bI2O_w%Ntq7Gmv)`#y%KmBy}2pX6Q#K2@I{T4$+=386O!$G#eDl'
            'IG)S!5q6$q;w0VpeT;+fQ#8>60SjC5R!Vvf>D%P|elhrfBvAUqAor)?qy)e|uTR~v4_zgo*r00o%Mh=q*bCeeRM6QGB#%_yki(Sc'
            '=>NPI1=)R^%!Hgq7~PSjy+j5I5_k^ZW`S-MIsIV0tkm2qZ~|sHBca>24H5HCL<29rm!Q4htv;^y5ESVe01LD=Bk5ofe01AIrCrYa'
            'DI#Xv*q3p8$2K@A2eSA5oFc5_SzA0*5+vi(Bpf|fOwry&b*Et4W+ery*>K}4hAcqGfkCW%4f(i;Z{**A8FT+*h_--OB1jaFvWnPY'
            'l1?iGtNydn$fZPgb=c{j4htX^$Gd6lka^xUY_@<&nPHJzk=<eYjdo50^=%*)#-MmnH=l&B<+Y}|2}Q@Mg^w;)=m@?ThhnzkSd)+K'
            '8dK6{vRDF?{pZ(UKPOgL7qAd`HcXa_;odN#|4o(?4koB_8@?bpnxX)uSK@BU0d(jYsC-FRQHbGR6zw)fr;H1b2GM#Hq{}#>kfR8Y'
            '5*iu)*V60y9yBx1f7pqhV!*x;Fs05uxfU-M@B{IL-H7Jl4OxSvTI8>?c;V{^urJOZyX0r@4SIh=7c!?9xZ|h>7lZT5!@(uUqu4AD'
            '2J<)=Ot*V`es+0!#H}QXuno@B(FD5fV1PawPL`$mQSanO?rR19=-1u1APwVd{|2#b?5W@ZC0wBg!?^Kn69XR6nX`%TZHf&42SDBw'
            'cBo9+X}C?;(FKr_j>E2mj%naDz2-XPU`GfMq#ehBk1WGZ8DzF8{K4=a;5j)5>dKD)HZCj-anyr}Lra`~<?D=IXLYY(vDDkRNO1~l'
            '*YhYmH|N8kftLKkcw_k!9`Jph`OF-JAm$t}<c#O+8vy@nJY$i8oh{*|BhRE~tWCL=a4)=vx1$BLZwQ?g(LE3t=+ye>%EtgdDW={9'
            'QUF3x{;6XkJyY!<U6>uCTW%8N4ed0HQ`Hou6Qm!h!(zY+$-(!8u$|}M?|b|E`-Hfi=lgk^5)SJ)uO^3iHFcC%QwDj-FKBNiwUqDw'
            'yDx>e_(!?LAO)@#ff=hxQsOq0uo5m5EA`cY^Q;Ow-|v5&oqqNuR`h><0ldG52l6JTrNYL94jWV9{?FUr8xd<nMqE+7zt8^*;{8To'
            ';D3_s)P?BL1%M0`juvd3#hAT{ki522i6hUy4AvKMWdCb{q9FJlUH-jyJUHncAZjeaQuHE4I+8P;`1R~v6a9kMD<kqi52R4cGa?f+'
            'bWirt2#L~iXo|)a=z4S@P_}6-*^?ekI(YXu4rf^?3f}du*oGdd(+jdvOOCq51`RHnE=fS07rY72j6HZ}?13fbpbvMAY~xYfgHu9E'
            'wNQfeYq?Nd@<%_9g3(KU6943IxZ3|~R>_zF*qKL-ex4+6!ORGNeHv0ETq{^T+m{FPg$XVCnGb-_u_f^*4xx^>1?}o$(0g~>J3Z<B'
            'T~Z+<UKZ5Q7~mx^cq~B&Tor*^IE#xQi_kfD7Y9BN?RH!I3;%Z)ZDF%;Jx2$Lk40zv3cyn;7}1EAaNcF0cFcjac`%eKwgddaZ@p~2'
            'cs6Q<*_KpN;im-ovsVVlUwSX`x1oeA{4(w-J=06ax52`<>_@N3k5fS~q;x8<Ssec0uM-McU>KDS9<Am1t5M5;`QLs$ywh($qX6CT'
            'Ch_l3+P9RFrihoAHile->bG0E#OI@EinRZI>-qlkS5Je9G*tzsR|-!17G*L{nu3<`O{ZBwu!>;B%Y}6c+|B=oFA+O@4M$=bqN#!@'
            ')Qz9uuyDqR@7?)H>aRnP62!ai`|feS5{iEhlDNgjU^*vZY9$pqkmeTjY|+6eXkp2dt;oUfzuw;tk1x?E&`_?(!H4mDMWM79d#myS'
            'xlQepv%>H#(W;l<U?N{ldXw8FH)_8Gbr2nN5TS}K+NVgQ*CJ+n&5VSGwJ*9C$6axQVUQnd_#Se1808cAdY$04dGZzP?=#l(<WiW0'
            'gN%O@d>S=*3hKZyR#CR%U{nzBzTB({{QsS1oQq2T)o1-kL1Ba2pd#N@yDp87OfUtcp?^B)onQ10F`*S(P7jC}y%xR{>)B_JY_Tf&'
            '(})t~_}cnT@sirYig&tgX)&T$#^e6QK)cb~mLzpCaO<aD6h|L{DDs^aCp7Xx<I*;FbH}ZK0qvpa%2xHVZh5eI>2iRBc}<6OXmwSz'
            '4_=GE!d6h4kMipYzm`h~Q%~EHIr7v~w&snCp-2`6H3r6GN~J2pG&|d(108`~?=i3k%nWub6U-oz5g0e5Uek^ga^-4vco|qQ^x?|u'
            '_vUNdH4HGt3#~X6jrZ{FK)i6<;tZWURTjG`tR5Xa+4U1qHC<82)O0l$(=bUhit)t|&%xRIi#&k=XOUU>$XCTOT;ff>a<7{DQHlrS'
            'i{AhM3mo*kF8Z2u2mPNZO+eY!*{m=cHtP&#&61ex%Fq1h{djr&&i+J2*9wzWvr=4D;~tt94~?>&r>Pnq$y<lUX5G5^gjTAC>**Vd'
            'MwLpYKl?^sT{JezmC#{l(7kx)Q8y!`^)}8HF$L>uDWSM=l0|S7h{`<0ahx?xF<y>D>+$-@VWst)bmOeP*=RMNdE|6~NrCL|^XAKj'
            'cVXhgAw_ZmK*%<<R2dPfmSi=|6Dz<B0zX<UtJ{?fJi1wqMrug1Kp3bapXaKqel1a<SyCR#pOM(0xa0dEX*E#6Z8EKMwM|TrO{^`t'
            'Bg_OLI)B9*ZrfmnM2FKv5Z(-a>Yu*rpT6~udp~3Pl}!lrP<5qYnB6LF3H00gAb?sUI@$FeH(=@4na`FJsyqu{X+~R4sd(hR!fdvi'
            'NcEw9sPAkT`RgtqbeqS`1mO#RrJ9(0;g3`jqA&c7DkAn!b-|%_QcB_|zkq;;GDn{Dl>fx=<VeYSnEF&_4VGbdlE$3({^PO_3a7`)'
            '#YP%q>L2wdzuc<wIF_3gk4U*iHSU(%)u3OwVKoMo+f^b#xmndd7OWE%FEXs&6|Bgz^ls!tmYvE>=Wx>fn<u?+rIIul^iJNB)bvs+'
            '6E+LNxWfh;Xvy8WtWFf8b}768g?kp54PC$<xk<X5QY}bMR>_-FOqgo=A)*mX;j2o33d%@>Ex)LM7`#({5)<@52Un{c%u+j|ikFfG'
            '$87rH(XH3#XpE9Vr#<Jl&<;6V%G7}*tDmYLtPfJJl3PbpG+Y_1#G%?>-rqMT94aw3X478yQ8>jo5vdqG|MKp<Ig+0C-}Ej99{R*%'
            'IHuuml#?}2&8Uq@aGY}Q!Qj~Y>Fn|xar^npw$hcoX&B5Wjd=8rCPnf!m7{;bsrM)xXDwFfsE3%x@ocpu-!072%Lx}!m4G2k%rKf%'
            'I>X{1;!XejVxa65<2VH(6OIq848uBa4x;jq9T(H#-mnb}VnIJ<!8#psuuGOD;RuyfO6V}WL4uYg4Q+L&Ku)pCgY3FPaZe%fDaH(B'
            'lqR0gj#c&%QYcJJb{kJr=2DdbTT>r#7odY*$paT)P~p(X!%L{DriP;~fu4d9OP%1A4l33g<Jvx-yxQ{qX-CfpBc(APlE;URqDB(`'
            'z7)SOESRPeNwzqr(L9_5Vwaojm6G^tszTAA9fel#8DRAI%;8W=+!*+@W$npi+*UhX0wy<{J;%LjJ-gOn$-EAnm8U}>l}{$m$M#<R'
            'c1s6VJLdo_;IMmocG^Ge9*e`X6F4|W2b^`0vhCK`@7NH>;j4MQg&!E-72$u_jb|7@!F_iR#8o&+ui*iO-R`O%9=adzKDObUC`#kc'
            '(Geaez%e119+wg`Dw`k^d$u#ttp#6Xv|8FMcz-Kp9?a90khCKm$z{fVvNm$gUS-M>jY2|+GM&*L?uT&7R)`_Mb8<O>=W!s4NJaTa'
            'b>csqlYNY=P%fQSrc>juT-<4MlNrfmtXGXgF^14Vi_*b+aN-#V4$=$K;Wfa6E7&xL_R1lS-+<1pdgRGm5=O|or?Ou>mKUek2`x{?'
            '%so_})&<ASE_7-z^#8X2`{>pzfiFMu735zvR>J4QUIjjT+^Gtes|A{UpIr+<yP1Zj3`$cdN6!(y)}B^%Rx_58@47UCohG9q-ldG)'
            'pG8qN>cPf;`5s96C5KB^5jr14oVme62Fl*1D&yt0A1Dmsp(KaiMrD1ZS*<m=+kPpE@3$wWBMuC*6E_)}nHFJ`w4kUUj9s8s!@{M;'
            'kiMV9OYt$k&6Um%t-s1>`Na~g_ml%YNLra;-X>a!ce+}vUZu!pbV{o7&d{~u5J$pPBuLdx`ltO9SZ}(BXEL>=e}l1mnK(N={#C~%'
            'EfdggleHw^AY5Rb8kMr0Dle2b&j^0P`QP8ha~=#f40yziN@kS<b9@_&XF8|}&nG2N_t(2ArkRvk+)+lOVb#nqRt;VCDVsa*706mP'
            'jfPDJjKu$imMWQc8~$#nB--s&Hr9$l3Ej&=QZ8C|wqf*71YT^5yw;J)k8+8Hz{#tgs!3z3znK%h?J0Z-UXOLhaSH;zQ(`s5F!U;}'
            ')C{j(89SceW_85XqH0>Wsf+Rz)wSXl^A#%jfOp>ZqsiPG8%NCVy$XYwpOKnr#Z{Th!|}=9JVJGE>E_X&8c4v2fW<uuX|m?S9aYX@'
            '1g5iiz!jO4%4nP|IU){1l{!H8z#BEr5h`TOAY}*G<x*QZYz8x-!>LCSMrytv$>Gv01lltSIIZ3u=v5KsmWiiR&Rho~`^^m}8dyab'
            'ECs^4j3?9!^VD+y6a_pO7z0RuZJ7bez4G%?3GU*R^k$GVhI%8+n1HA5QKtkj?@Lu;7SNZB-E_sj5Ks-ChHIv$t1?bpsS&SrJM-)L'
            'Yi$Y4Q-!H~U0xpi!g;&szFIxJF3K>aIGF0|@A_U@b+2r5FOHh2zDqCkD#WR9AM0VUM>+h)fJE_S2_iZYhuh0POM;^vmBVeVhs7Rs'
            'P65z+8<&CNG;<OL1DKd-$k=ud6&%EK<-e{=Ioza7Vj());@JHFzF^#&vQ4C684uViSK0OoHtZ`L>)OnGNb$!YLCiu*xZEK%=PGYf'
            'JZUtIVkQg<utq(Nfv23_Jbt5r*woW&O487*ARn$ZcsNz#mjs96D%zMcI+_%$mD(-L;^ew7uUmC}V2)_)DdmaB?_-!5{V>%n+ARiZ'
            'nJ6$caLMTOsZ;sU<03wBTLube3FM6_kQtsWHsD{yQ6zXJ6{0CifR|Gnt`HlMo?BTtIB{bX%ek-=;*{`IGc<G64m-KxPNw{MI~BeW'
            '{-&zr;S^ZPGrnP*F3-;>pzgjMrm@NeYsUIzbMF@D5U!=(>{&5%j&FuyoIkAWp(HMY9rB~=`sPX^EB4{%X2)Fg#6-eI>~<k%W+{JH'
            'ZLoixC`^0&pweHOnWdX~)=Os<VbRUVF(pt(b617U1#a$E?_!k+!i=%378{{`-K(nJx%gmq%%2*4Oyo8MbSWqp4Rs4s5-{-(0|A%}'
            ';Ax{DcBHDQ&nao$jFqj*Ac5cqNKvH)^IV{j&qEI~&N<EXP|(s+RR_iBmmhb#JVvpFr%7EQ-eDbNK@`mNu@A)>_Hb?(4ZkCA>^YXU'
            'OTsw@W>r-zkWSf)Ii~FPQ6q%`lh-J}&ck9vIh9u#`8RXuF~%|n<=6*FjgU!H*iUjXPEPx(v>`~gv8%EM*{^LVY^Q%#Ym+wpRrBen'
            'A6()$?NsXj&FcIFs;^G!ur2eI++iaRuDZ=cT}?>>l`2~~OR&ZdAYG}<*}0AEO<doat@>g?psq-Dq;Pc|&qU*)FpSD_RIe#2;i0^k'
            'N(A2*jiPffN6&rCzf^=8!@c|g%c{#!4O}f`ExB}shQ|DQ7Nq!EhDG`qyJz{9$_rA&6NNS&h`m5h(%Xi5BJ-YVyB;8=;;sjvH^zP@'
            '(Lp9iD|paKkZQ?5rhn9iJ-WoTIlH^T9~i!k-pXnQbTOX)k>?v>2-siArGn-Lsla{0F0ZXyn5ZQeKXPJXFXr0a*_>dmv_#83*3LeL'
            '(KN2NkMW%H_DMkXj2hgpQ#QYSim6Vee#X`<<14QCODC_C*C3#F2p1XvWLIC%#s0(tp@vQCRW^0NG%Pa>bRba_bhCkd=hKaMJRWS*'
            '7{zWv?$5KKZ()tkPsz4pZaz$(v0l$g%wIm5wn^R}V^z)ktS13gxf{xj*?YpPU?>v=hPto<+Tu|i9UMyobP`~(ym$&fb*1=mFrV`s'
            's;n<f!UA7OlJQz!9&eYtf2SS{h~?@^+?0upstFrP7wqKx)yytvpsg$RY7;Tm`fKQ<jF!#JLQ%5(QC>o%$hb&%Lw3tC?_-DR9kr>}'
            'XM;MU(*)<V3Zik0t_mGzl}%fJa$L7!1TTkoqrs1ub{cAkbkJcRAPq+M2~34PPr=zx?i<DFbj;k<m7gkeFdl=?B>ca@xH|fmY5uc8'
            '0a79UOR}JI8)oG^_9~MVHQzFoC??oLgX}m?lGQTf&Uh@7fT(0qjehc0a_7s+p;gyf70+^@8Yje%r*p{RbvJ{KYBzd_puiV_b&d%`'
            'o4VU3F?NG6{&lWpwJa`0p8D%jw@g(}*jrV{+w%1v-K-U_|2ACxsXI@ZWff@a*_$__F+9IKRW6E$XQ!t<<y3j0%ODO_nPf$9-8SbA'
            'HLvvKJ>F|7;J4;<@8nLe?qTg4J|=m#VJGpU+@ByLbCzd2Mwku<l4urg{K@8tI;c+dp9W*h+&Kb)I)Yye8LEhh01PIa$Sqqfu!JQt'
            '3r;eMzTRz}73sf{;|POE`90LbsYV#F0r$DiCYIfp@B>l$z&2VE**8UXP3BFDM(zQMPi=N!Su9;MS+cZLSD9<ox6S(?)1IBXo}zDO'
            '_(V9Ldu@J}1dJ-{G{W>s!}dD2_2QQ>I~aXXTYlwz>ZXmV3wKZ(<`inc@{va^?$NEB9Nwzw(xwG(jx3B;fqf%|x@3nJd$oVPQyk4k'
            'yz6?qcuBfYj+0cp&h|AbW)EqUph68hxxGO)#Ec?{nPo8Y)l=ueS@<7hf;}$$hbQ1qk9G%l%Uvy>s_Jbv4E!4|^78}^5un9lnXOe4'
            'I8VPJp-cVcx<%hAYs|;`Oyni9vK4}Sc|g<=r{CkO^?43k2EcT^9$e*DaP{UYf3_!-_)e`=rh8_0?>~NdckkZ$`PsR5-W~M3<Nis1'
            ';GMreL6M5i8`(nX9=NLAQK{WTwZn3mltgls-KuA&XP}#4xCKdU?1K^mC!}%%vdu;B9E&|;{!0otm-im>?Vx|s17b}XEyG0+V^U1n'
            'uluGq_?7zk(}2P-robQO>V5z8=k9U;$b;VRd4Mn?UC*n?v&mfy1>&TCanV10i`^>No@{@4`tJ1X7Yx<W*B4x^6;~i#y*L4M7Gqr+'
            ')<m&pe3*$5s8F*Y*-fdyB9<@??}C={&!BEtjdgI|J-z6|tUUhayZ-4Bh1#0XYYGcmt~m>?zys>^_|`1yzKHLr>O9MHKvVc+ws+oi'
            'kB@)s9=<Dnj}Ar^>NKmK9;v4q1)9bHm9tiV+8fuz(%~vg`*HtW&wG!dGv}xIq^!cH-Tp`aYQObou6gSH*1CRpw%>eIeuh@t#t)(U'
            '+XxdFUtz-^Ky3XNqHmnYL-?;z;3q-yuMaJx8(0p#1#O_uZ|S#J2c2v9=eLXRYR7I|{qS$sjUNtZ+-2zT2PkNrIgi_Mum1DjeoKFI'
            'cCWwNFvWD%GK#pGYjSnH%nOaxXN1(8XFIR0QCI9+Ghu1vq%%Uff||K0h3av!J0)GZQcX%3C6unC49(qKkDa??fYC+oqzi}X5b)_&'
            'P|{*d?oT`|cN-wr{U!$Bx8i9F<k=e;&4YE@o0yUc3(qIv%}tQx3EL^BJLtKntx>{Jj9ij=r&0_GU`*yR<q|kVBF>pYY$}9KsRCr?'
            '<EgB7CYZp;+;?ePmZ#6uywvToPVjD2NUvu+#h8c_tb%2g`E1C#pTsOML?OCy@x)?YCpf|w7m;GB2yTF!PO5gl#K}x&;*!Ne_xR3j'
            'Zw!kQ=_&zjgzJ(5DN>9GvwXzatxgD^*XxqeHb$Hc8RTNjCqlWyRS6I1RSH%yZjXs?mp7wqnKx;C?Ko8!UpJIZRqBSU(ZT8lEi;TA'
            '=>R;DHDSm;NJBgL(2<o^OzR=Pl~<x>BUUz%Gy{VX4%fuUy=kzzo}Xmw&WXRMIG82Q*q@M?poVW&&LDrQ^}zf>5J84tGK@3x_+DNX'
            ';7r@}XT#MOr349_*USB9P4RO71uflKBKoFuaR`tOsjO$D$HEq?2q`h;qsCH2*8AUf3Kl@jd5lJfoPd?alUeFLDFm4BYc>}J8fzJy'
            '4lLj$jh0I>#qZqvsW3_^ZJf2FxOw@_rzfS^X=!H9|4d_|B884}Uhv`8U!_?EXrbv76JxiAVn3Ao_^!NqqA(i|WA#!GBe=W?h%#$='
            '&nerwl|_xJ#D|@y+nkhe43p%F@AseM|GdEedCAyD9{mx}G!`v4uN@Fid4!iBfVWg=1g`)pd~u8IW5GvFT#K~D!RQK1RK1x5{sdiX'
            '@#Z}UCR^c*l&|zxnVpq}9&wql(sN*Mx8W$vjC_@wAqb2)l%6G9KqM#e)^jOaD|ds<cvY2RNd;R*>&C)QW|&AQpETDmRZopN>Oh##'
            'PlU=}V-+|0C7tS0Z%Mu|YfdS^1`72~wKg1*aGvtKV24#SU(IN^rE;^;H=00HQ@o-9Nya0A9Sf7iAZ##gt`3y_DUOIU%S>G@DThoj'
            '7%*A4x+M7Tv3ztC%>6Zsd{lHRQlT%$2w05RZiKR`ri((yPlLgGF*F7mid#TEEZyMaEgnvEf@5MK2>pRVoGjf0V~{O;4h^5=bRI?='
            'Opahx{m_W@;#um5Db~`Z5+9LvT9<6Atz#VN=8=|Sq$x{%_IS9pil5`ZJnz)t$2Q+n0&vdx9NDFwTNfHsh!e6!Dc+XKcmk(Fv~vvj'
            'GM?d|zWNh1p7UQ@-8Z1^I0ecH-;#sK@04$wO4hc*@mu&G8vAX`DN6V2a-1s|_S!99x=Hg<Ls#%=ijrzks^pGEpL9t*03I*eKpwql'
            '#saHe(0x&W!e1u5mHii5O6xBnn6U@;o`KK|C(;mb3Okd37TULEYqAmJAXVQi7om~Wdhz7qg}^utkQJ-5tmBc5p;YVI;jHrM)K*@K'
            'p+<&=(>7<H0A-OO`vZaYH|GaKNp*f`{5JUxRMFqs@UQ!WoBrS!5+07y*H}zLmtCC*P3T=4HKAqWnQK|gRNT3w*cEF(s5mDSw+*x>'
            'lZfb+W$gl{;z(}YNO{rh0+6_(bmIIcfTI8lc#<Np{+EN5pzp1nF2G_dc;pD6GXzj@x_n+F8XGa7!e(9t6FG)kviELvV2jB#%5Gd+'
            'Sz+a!CYsmhSe`S)<XRww?U!`j0nu?_&AGO_M~V(~LH#zp1$o#Ux<vu!tOK`I#QOwm28o96%H+53r<wFbmJj73o0u~Jxduq*C~nWu'
            '=~`d;H%7?w*2Fv5YPUdt)3rE0J9~G*`3QU@xO<j$NYdWDe46sQ{_^}7xE0WdT$k5LD-xA@FwJ6|C4BcDQ)#~Dge(-=uQPQP0@hg-'
            'gh8uR*SwhW)Oo;U;dHdy9!33m-s6FIfCrCNT6w)sH&N6l3Z}_vk@}~+#EP}R0Q<lF_FK@FyO?woemW}Pv3@4BAp85<ezfj$tSMum'
            ')(Ck~rCG7`4cVccu11ZyzZgw?u{aQog+BG-yYlCVc5sWdA@`-&XyZ*Ee!SWOKx^g%7B&IK7gq=J_+5X6Ym4{(7_Tc@sGgvd%sEWZ'
            '%K%f+uTXG9>GS;&)e*@MFdRb(2jd2-_VJa*t1o}0K^$kv)AQo#6u!=WI1QOPYn6=;v{#}l!@M;;y1Y<p6PJTS!2=wK)*r=Wg(OED'
            'T@0F{^(R5)?(f4?g==4;Qycm=k~1aCq#p$(oWS%GN|9{JD(5J!qLqaz`z2K45k{uce6oUbAf5xpUm#dXygQ*%8PE0)_V;OO(qR8A'
            '&5TB;pqDO20O0*U%XThPj{agf59GGwImBgZaX|`7i?)hQdoX3&vWVA&;+HU*#P_L8TJu30{)X8{T<DOJ*I>qv$%gzIRDDnbO4c9z'
            'JN(^;zjZ4HJRp0%Yi&PO{-X`mzTtF%K!;<0?>J7qZgdmOgVb?dbTH*18Wt0m!`6ye^85fsG`MLy1FW5E*~Jxmd*a0_U}R7D+-iw;'
            '4I-Jn416L5w}fO8M8*a_ZHgC7@yrw*yp*@LBO8Bb_{75Vw9|McTKukk9|SXah7Jt>kN6`r{iAZnl{jI2>De>=Dax>ve@VBm{+N1Q'
            '_#O{6!GSWgK;~}D4QnZF{=EHc5l85!Wj}=f-#{0!?hR(zu5&`hnFNvPPTerqDW_pQlC!X$ZJNZNH_t%g5|9q`x7L99i-`>4FM6Ux'
            '7g~SPW&oyO@O?B^RuZoY*IP@pf`h#)a|2Nh2Gr(}B+DB`F$(4}mPyItCZ?2NN}#DXqi8sZ!kgQyrHCTY2<<y-ETkihIc|3R&ee<j'
            'P}Di0TVF8RKk0;C*U=y%<xcZ>?9Z+C)*)ce&ZoGG&5CKrHXS_Ij3t!&jQ6d2KmzWzTdiK@tP7X|i%*%#9>%x1#XZz)`M<+HYDVxR'
            '*tg$79zjw^+yLt3DL|Pa7~^;`3ISKlZ7dVUXYMDZ|IniYdu<bGa-XInRTFL1E{(CP&LI}LH6f9eGO|lexYGg10gI&|f9TM-P@RNH'
            'z~zX)6LG4xPi2Bh{8TwA$5cjsyh;)@;-Q27*pH;bGY@>qCU)!3r!D+k7DQ4q+|$ng+CSg$ipSr%uY7yyCzKs+2D57LAE~Uo5q!Yl'
            'uXsj(O6eKa)8URNxB!KpPQwrE#LmBAO;e{0?H*z>_bkIn!+msg_cp_m{I19iezIDCP>;}rFGeY7L!O_G!%&qnr9$yp=Q$|`zhUO1'
            'l8-zCE6oqWV%ZeDPIYDNy@6dgj>og2`!4`IPBE!^JmwPb<C(Y%eR0NF!~b>WVP%*mXD{+m;G737Y9+55r0n`Oj)DI~@#^N59XMgP'
            'c{_GNNru@%H(Gkq!X!pI$YO=oSW?We45Ta-KH;0XIn`5FA|o|A4x?2rGy)J4yuX%}?8Cs&ahZdoB=5mWGtUDVVm@WKG}Yug&&cet'
            'jOX(X{QADh3uLQ#=M_9M6K_EOwuwr7X$oM*t!X1r3};emEkDzgec?0mr|}ul5xSXdk-xNu^QIF?zRl%Py$88IW4h{cM~*Lv1UiNp'
            'Gb=Cd6Hf&B%u{p?KC>m^N~5u{5I|Wh_C;{i-v=@V&n5Z+&!RsN2i5Oi&e1L1tv74Vr#ltKW;_5roPSD2t4}a-WAvT@z?K0k)W3iy'
            '15g&kV*gs&-Gf@9o{zGMV4r3U=?_+hZ-wuv?O0n_>e)rj5-qJ_!xI16(b!Wx$T#e%FF3UgMF^-;7*i}72d_xm!`H6`5tm?6GZL(x'
            't_nqiGK3+G5MhSUAxD483B;U7>Y>|iJZ&njDxC+xvZ0TlYaN~s25tBj0O%nLb*+17jk2{mA0Xy<9;ZP=HFbAta9*25KMe(}_{%)U'
            '?s4zvZSTA)!!8Hq6LpoDb<vCO&~_J8R`%5N4?XOkA71tc-j8qq-@)L6pU!(1Kb;+;+vSVW+ru+>bN0(ATEC#dv+u>X;?=(9JO6>x'
            'I3_-zInR4cv4MHRf#CHn2K|#RmJUL$z`jy!<&oJ1aue+UjhU2&^Y{$Y^Qo??AeoPzN8v*FBV-NIwwiGnX?d-ua*VR#O%WxW)C$L)'
            '`y-6tn1~@J30R<?2zyo!+oGQ_?MvC(fft-0{7}#!6{l8pb1EYT;Y5HSz$sbLWUP_|r%M6>vNg0B8}5`UB^G;)p_yjklhv4#^U?Q-'
            'zmlM;W~;}gM5_cUw2S~k<p}|g=oVQjZz+5ZhKmV!xs0!jS53aIP`+4(O^yZstvIc;;)C4pO{2P~rn<1u&Zc0z4Sbn-GI<xl!#vBj'
            'eH?X&G0Y4_O=*rjcv4%@LsZ0ZZjJygKPrq!w!V&>Ah|c%wx@ilR8UHjkBaC^Jtn%CC?6^`l;TN~dHaLUwVUQ!&XXk3!;UbkPq4am'
            'rXBXYSBJ4O&Vl0C@z_&3o^o7=^K0+CcPs|opHBLxZM1T(#_<1<SO?h*1Gp#=!eo`K7AsiL4Tf#$oF3(1O7wOTnD-;YEFf6x@)PoY'
            '7`Z=ew+>d?J%wuMzffV41f{xgLS3hrDC|=*vJ?{cD2(I`PF2WV=(<`Wn#(Ll<&&MKpLuvJzVV|K3`?snM%q2*<b)7cW0qsE5@*Ae'
            'CVN!bLB{z(**emghcP3K`AIR**xNoryZ%4>+KN9UJqmL2RSrL{?SsTY!Ji$K2qtww$;GvH>M+2q5$7e@;&JMmv|+!c#L*}q0p@LU'
            'w~S7%rIcpKvb-cBdnTGpslJC270Af2QzbuY<V*d#AV)r1;!{Nd6;@T*|I2j!$~-aGdhW<%Y^$DDt5D_Eio2m9+21Q0XmrbF{`CR_'
            'XeqaJPU-pi41h<V@*>Rj%auiJp73;S4T{_}c83JCjtNK|9at{#p)>F(v99WZMvhn!X11Bz=tOnozb6e--U4(niVohdXw5IZd$M7$'
            '`M#1|+Fu7jDi0{!?n9+%O~@0js;RsXsGmn(3(-S~Gp!kibI5LN0!lze4eHoL-U?Xj#zx~fo&yXc54D1MeLH5U+F|*r=0?1fNA-YY'
            'cr`^%)h$bgU3og;9BF?Or*3UB7>(31PgN1vums7i^PFZC(U{dVF+Utu&R53y!lA0DRoXF%))Zd4n|f!PdS_4d=0d2o-=0+Tw$r#;'
            'eXUoVLj~`zJY!7Bg^K1n4;3PLR?`N5s{HBvlqMj@hNuAQ6`&`TwK4lm(WqT9XdURzq(`ni!`D0IFk&52O1qj%<JSC{9?Mmfxfts|'
            '1{2CHhjfQqygr0B6(m-s#*!4p6@S2sk98Hnz0$3&>2_pY*`^u<&|$x3$g0DBT{5)D`($&@Rg~FQw(!w57ENroDeD$BaI1`NvTN;?'
            'SM_I9uO~0QiUK2N$dcXNl3&7KJL1)TotrPiy9#L6tVUPLFk5Z0<ZI<!u0MI3_+xpT&GO`zSl{QZ?f0y!oD=v{*I)>QNzj_k`8x3T'
            'S_H@>OeB$WI95o7O;NpI<>a|vnZi+;n|`@nD?`}H<;8$<*N+19=SD36ty8oPN~Hw!75hu$7yfq^>B?Ctg{x9}X6^o8u&9Ch_H*w-'
            'S0R#BDRQcUViFpzkzi^XgvldVQrT0{*?7wVFra-Aqtyh>fn@zNEZXX>vMlo&B6eBUt*7F63nxQj$+SuS@{JmbsY6QGAd;<MKPKUm'
            '^;8-~sf;}1?-ESflZN9^01uft#q?Fw)TYKX(UOKs4L{+ghHvca2{Dc5)?M;&@2zOD^)cD+lc=#)Bk#pu<N)<w(&<4TXd@p*V-#i~'
            '9G3@Lkje}CX%w$?7gS{F1sSy3%Uko|LTe+j%W_g+yhYiL#2{-LOR`lwB@+O<f|k|p^bY6v&HP(9bq7M?Hks(eB%ZG*yDE*oTQ&Ob'
            '_2b?H4E~6YE(Ul(4&JnwMde}^n-`<0JFRE2)p{oWES&kgp76;9@fYNp7PzJpV@=tnHD#4+%H*1=$vD0G?RPV~@9KvCD}Mh|z(oCQ'
            'VEylh3?VVFAYnx+XhTrsFVovN+tR%838cH)I$la`^GW9LdgH{C_*eI#Cu|X)xUEmkm#r3?Pk!x8ZGG(EX^rivk|uR}@>4}k>NMo1'
            '%9_+^ip72f^MmeYJuiQ)U_i;gX1%yB-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWyx<r&YN*!tER~FG*x7ltr2FP5JsEAHrs5<%~*)J'
            'mA9Yrw%yBaG2<t|x>I|{UNf#5LGKg}zUeCd_fe*Lq4H@i_FYFj+xcvo&lZ&ZCb`eo(0{U;Y%wL45-R6onI$*rcmIWKJ}3`ch*+)t'
            'ZLl};LRA$v`(?xa)7`N`9%j|wD%pt(;4YZ^HrZLr3bK|*OLPrys2p_s^iZ*K>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_2Y'
            'Pk`la*lcMy--O2YrvZFb->^UNjX2`MaO6Ej+N8_?j;RiWk=R1&bM;Ol+Z*jFT-nj=xW$?<8D~{;vxCn=#u6Hv*!Wau5adfF|C5lE'
            '(7L`u8BNKuXS$S2qh6<qak$m+r0B^iEYVqFYOK0N)laAA6Qk99#T6YeWzS3ZH9SE}3dV!M*C)PE$wn9cN9QT)Ho8}N3R?R3*UII^'
            'a^{~Pm)CAgPIPv*B_4YE=7o3Fq@}FVltw8j-LPxOXstY=9H)Y{Q^GGCotvW!MyZ?GX*<%<dUCRKI`JEZVN841MTX~{PK*{u+NqnJ'
            '?-V~I*Er?r*0ekf*jOf4pt6dj;Y61DX2&;}6BhQC_wpm4B`Ftp^eeLEx6*rmNq*u!B~Pn1o~d-YY}%8RJmh+1Y4&28hA<6`c=ie}'
            'BHTN`XR?i9ZnI<#SS<jgSjj#KX5&D7j1x36Oje1|$%%IZG4t0G%my7MZqXGeKY5tYq+Elos0mU!oMNDFvBH~LVyP9Q5w+u=$~s!v'
            'k}hIv926>w16rxE+2x9L@KAv(c`Z_Mz9Jx|_+~=&L4gniAKO-yHf@>k(Qz`RoXh$y)2vBDzA;>(v3&n&TS9p(PAN^J9z1n_+Fr)X'
            '1~)Dy3fbDDMJ2SQ9WH79^i<`JXNn}fcUj$GAC?2tiFRA<=ZOUJUr^;iGa@?_>MA4Ty1Z3kZAuqJxmMVGuQSh64hA(zEAG3U=X#eE'
            'T|T!o#Zt~`1XCxTSuo0f=4-2lOsD#B2~PH(X4wJsMwMXF-;xt2SEE9zVCH-r-nUU=*KoG(N&rR|OmLHcv%yIt81-oqffdWSPenhW'
            'ZzC_!eR1?o=9h5WM%qP=(<vFni^)E0Pd53rLTVhonTL&|*E>dfL1fSL677%`HLSUbOq(zZ)^;wd1Dzp7e%EY4&S6pfPyayl{`lgV'
            '3|bE6lv#xegaG>SwtB?=@G4`B6|?XzAuYb)t5fUYPBCNkEe1nnX?aj88X~iwS#MRawcK#>QKCuG(&QycezAkpjMwmJV$TE6gF8+Q'
            ')r22mWQ?e)uHr`BnehydiXBL-QwVS2J7tR~<vTb`R>0K%_siD!)=yFj?_y+AV>FqolSrat!|0}M@cu;Hto&qRq8jIJsll{;vlfkk'
            'YS^imut(;)Fv9G(RXNPAP}So@<5?3<XC&9Vw<lh@Um*j=IaL(k@}-=!JNQ6~g-+O=iWlPFzDCw=vTW0b70aQ}11eyUa~33?TJXeL'
            '27td9E|7o}m+AyPW(|^B9>KIu_6efAgr$dt!?A3*#Xjl>!i*%`QSuDUw(>$oe^aGLWQEVcpm*|qaPisva4Tj;`mdSr0?6!Y5@w_#'
            '%fcy`5s!`l@;sw_3cl{_B7}E&LhwDaggOU?U|xZFMkxnDeA5*$zEh0E7)nX}yEGJ?yrF~%)vRE^oc#<MUjL|f!DWDluT{=Fm5sU5'
            '4+RTHQR0u)2$q?xxCnD>DH`M3I2==cl65L>VYR6W6#WHtl3^tVbYxu^9SMU}AOo?QlD1+P3?^%)KKwz>azVkRLfX+NM3|#xFbO#p'
            '2mw%YCSJP4FC`SFQX{gup;6Dn%pO<J52g;5#)uZd8i<@6`Ub<^!UOSu!~G$_JKA72$C5AbOqOUhMt2`w$2Ds#su?uYIdc>&?L>B^'
            'soVr(lq)_8xxF+#?Js;7M=nf8V%fh$jw3h~jv-cO8-p|TZ>pSPy*FRyB~~f~0M)8UnT>(2)bHyZ3m$0W(;%>mgj0zS6a-N6*F#|l'
            'kL#AzK-s?X!f>lX6H*0hR#nBOS~v8#5XX@F((1a`)HW|?8^xYlpQjd4<v_Z*ysg%8oYMZAQ~=f0P3M1?@b+={x+3UBf+Y$Ub5`2s'
            'uZ6A~%7hK8AuU2Kqr31ww#$RKRo<(%bhmM9jkCkVN}_DB)asM9<8=i#U9NuHVnXIyqoxpz%C&^(GQb+Xw%(9#XO(j%`&3plA-nK1'
            'ef>&<f%$_(Hau3>sDSD`Wm;tpqpBnE3FUE#<bF!!`C=j6LK&e_FmrwV+`6*6l{Ixgd7e*K-`!&7TNmd2bTC$z@ZDxoHMoZMx8kmE'
            'Xj=B@kTu73!c-H_ta8D>Yu#!}APy8tQxPrN1^_Y`XhX^_vK4AmLeYmYpI5b@=$5Yv$b?&V75<c!^N&Y&=gIhgI{oERS*EH~EpnF%'
            't8rbfgJ;(}6;G9Bgm<%LG2C^pcXldytHo38`t*vo?pPin#<3$f*tNfnwk3Y7e-~S+3ioE+c+HvRuga?PvV1jzf-^4y=QfYWrdsq)'
            'v2&p~{cSiN=^|@Y=q(y}&i5o~(y4943`i*jp}Mb&h3e&)WB(@6EE%f~RurHoJ&^6XvO*o^aA;_iRF%~(rDgq=lG}P%&d|533s>e4'
            ')8!h;)i~F3M}8DZyVJr?G4x%>&~pJusWcn{#-m|S0;+$+BxxCgc(bM~hyiCjZMAeZ0M6K2s1GR-+M$S2&VZ>GYw&8#UCHL&s4Iyl'
            'eaDPINuCPG6d)OolW=5(fz!)Iv0&rV0ihc*sxm^VdQo;g?A-qC3`nhi&`Z7A#k073{rH=4lQnh7AOY<N28DO|hoM1nQH8c$zCqF@'
            'Y!Hq3lCbbS|6ULZiPaK=M`7e$CRCGMnHif=jMurVM%k5#XH|}>x_Wj+RP15CAWWP~mLbX_31YF#y?@9CxpP&)hzjPIRcdm3s5C5j'
            'WPH<~qq3PFvRtpjze)xye8^aH3)@DQ8@ZUK4UUvhwlsWhk5mY1r-D~?TVzl9kM2cfW4Pu5=B3Bh6$6n)KMcr1D@VYV_Ua5orM4CI'
            'iD`L?DN!Q}Jso-cxLtWC*^aNb(kjDHhD^cJMGuDtnQ`XXMcqmWQna0)8vz2;qZBA0@3&><8aCAs=g~N#-gOGQVN6;You@%CL2gW&'
            'Lfy2_wZbGhSUHV)ag4Uvzw;pb$>Z`nF^mWGB=Td@rV0d`qVeliFM#N$<F;3T21%n3Kb8-N#N4i}VZOEMYdn=Cui<bJ5Hz8@KR^4a'
            '|6_m9JHo$yJU%;ocMQ4%mqR%054xxumckP>Y^}9)9*ikHIyI@yJy{CdC^$L|Z%ngl`TkC$JSWE3qV{vT241=FkU?P4Dxk1Nwsfl)'
            '=@x@=Au^D5Z9d-Q>}s7nS)^(I{1KPQ<(Y@P#v$J%;a&}E%B~J|vFJH)MtlPto6Dc!l_{kQ#K%~&a8G3M(p$=ZXYjAaGQKQni}x};'
            'T0EVGV_-!<$?3-Vg}7NkLriaCj+oxYtNDaXl4u(YvsB#;iD#GuPWdk2nkX}~K<Ax6_!C_UL||@plriE(n5imd3$jv@kVBH$J&d2l'
            'X!MK|IUsyaPsyi*dy;@m{RK!AnA-#9jvfWkc)proY@Q{yJRLJ{+9G&ASaic1q{qAzLjgN-yyrew71eCfkwppo)|<TKKnWW;JB<#='
            'rB%w{Q0E30Y8dps5xjI+U{3B~_kADrXaGE*2CyF_So|GKS`@sdw&oWNgY`FngmZLqq5S_`z8r}p8qHNG7m*Se&NrM$6b#Di;Q-^('
            'W6l-|;wBy%L=e&$=>fSGu?^97Fd}Uf8k9k=2jB<v%?HOG7BmX_S8r*7c9<$IZ33IWJiwRB!@=cw?+9sYzB9^hLN%+)Q_`a^!Uf1y'
            '>!oyq^0DRt7n<GU<YKiwd0r03pj*$>5_0-uatAr_x98}12Gf@IgeRw7V<mr3yYQ$C_7_y=B>y@P1K8i$9Pn=)Cv8K{(HjKRk<Y``'
            'gG#f;)lw+Q&EvIN5fRQJG~1m0qd_k%Nr>Mo214$Z9%UCrh7QE7AvJ6dQ-OYHnoj0T_RjRCQ4@5jH0Tf|8~e-7pYTT#-4v=knZ=y$'
            'QK2eUKGJ2Mpxf&_Lp|Q?z!MVrS-j!7v5VP1F#Re_hBEl`?bHw^F*I*=jagS4VyMszT2MhKl(kv%E+86FHqI)uG3;H3{ufAK0!S~@'
            '-xtS(2{Dkwx1#Y}eEY4i;Oe^ge82s@_znka^GKMjT|C!Em&g07FruW@1~lkhp@t1SQTOSj^9HrIt>lc3+Xw=|c#a;WlBM$>-qfd*'
            'x$0p-AFpTdTIEv`!k4?!{c?A@U*}TyX)h<8tF1maojJqY*%vVr*tQz#d43KhG}Ou4Fk;z7SdR9o!69`4B6Z`LaUvUrl<rEN@<RJm'
            '=dNTGrTU&5Whu%HjA9pCVTf`d<(aPLvY4VY=twbfD9=QLk3iEfk|G<RNuY&5@s%M?8I4&ij=_Oh8I@;ou-9T}Y<a`%Z18CumrCeO'
            'hOq5xW+yzO$|A8tE=g7>U3`W@Eu19vw=5MZ%fPeL@gvs9#Uegogbr7DkItpF!<5rS;zf2UGgXcN`><>>U^Q9YRT5;Gdosw-G1jn`'
            'p`}BSsg~%<b0wrjA0&eQ;o$gJanbvaOU_mXf1t>4-UC75^g;<_RLVfeNRM1t>WaC~mA_Gr4XN%puAVptR^%f>&%#%57se~b1yBiA'
            'VT5u&M@lpeuP{zGPp(Rwxsfe-Q$M02MoMyK5iaxR@l9JV4~x!YZa&;QlDrFL3l(Q#PK7Lr`BxK%i>#96`H)9?NDxD@&H<nD*K*OM'
            'l1?fjAyl5e%Cn`jYu0}F<^Hq1m-{cY6G%?%siumwfoV^wd!QWwp0?Lm1!|d^1HIJVBz046>e~dXe7nB0V@V(%^ZO4U^q2~$#HX9Z'
            'TJ#bHPR)w+LX%Kqn(Abv`f#b5Rq%~%%5-0JPkOe)r-ZIzL;1I4@oi<+fe%~^&fZ_NWj<pvSO)Tt?CDb60GZJmgXLxad^|dutQ+ch'
            '7MXM|1C-xoEZ=R)6%kKRADjB4gz6C&Ds#bif+X{z*sIRxD9tE}<DhDyej-ZfjK@1UkJeXP)wf}6UNzi|)hBey86!LNPF`wGIj)tQ'
            '_3~GpN;fk*sg72YI_zhW%Om=iA(W^eO(lQuKQ<?;_Ajb@+N_X1I^{=&@juB;*ofwn@*(m7%j3W1j!dB-UL`REox%-TIkj2EwV3je'
            'FI{vPt<;rAX`tmCZ)r9TQ{OqiWUK0|-d1#1E7u!PaIkTBC`zYD4a#s(JM5wRDZFO8R>k|8G;0nY+6n+0f<yW36ItGjSW-UMqX2WJ'
            'Ki!T?xt5wCACAUrl=v`PU3<Us8j}gprmM>0)>yBlA=jLpcAmGU^tfYPv^|Y6<|Q?~F%8amOs>=_$n*d@>x%4s94ZU&t!ZwCqdmQ}'
            '!nJJS!cUqE=(vS=@%LIheaTvym$^~be5)Q!8impVOmUmyMMbi=lnSDjGT>m5_W2#Ul#`r{HxBZVe4+(tTbDG|NUIhZFE_vNJK?PK'
            '7&QPbe)s+un1fw7$k0fdd(ywS=%2pbW?#1{B>Gi1f^Oa7G$h9a%gYJ`-38gW8-&&F6B$r*ZzkPNd7*p>zmzaf%vHSXV3?H2?X~t&'
            'Df=W=u_aOE%o5Mabhny>I1~{uUwf)@d{Z`ncWUZ%K{cPJ9-Os`Jg9Y^K;<A%J7lOj4789B$&Lnf$v?Fs{9KM)6RVo*nBp7KhX9Zx'
            'eRsHHe_NiGYHo2|m)rJ<UV801uT8;Uh{;RA&|wAV=VEyEYrR9(V`Xe(MZXYPZ@rY^$42<<Cl{G9=1_{{W(zSUxKQ?m0s$=<8Y&{O'
            'S$*i#UN9GHbL&l|_cb{d$!vW=Bb{?YHebtL<q_FRVHO==tA=UJz(Ak;Elc`Ee~IF3i`m0UlwbWPcZzC#gwu724wt9A2vUk9`()y6'
            'pJ6*ta;M%p*|Z}7)zxMm1Zao<eZGvZ>}4I>ZdM-JIA;cC*dh0Pg<Y45^D`*+m!;g*6r3QL6K%QEMVg)eQ9bUXyK&V#Q%buc=en`E'
            'DVm@iD!BKQJE60&V?jv4A6tCL6iyoTV0(M{bMN%H|4x9=bJRc3Mzk!r31P$u5S1)!i?^%0Aex6W3j2W%;l-$*F8moso;$T4m@=65'
            'QdNjT9FdsOKkpx(^p5&yftXNairl2f>ajL&MYVpinZ;}x?{men%JeL`h*ovb^5iSs%fV00S{c;+pCh>WOi8u!|Hq@ms@V*KQ)Xfk'
            '-y|!_&2Z~umH|vVLdI10pp<$<_N+fejy^sq#=T%xQw(4bp!fMREELn;cmz!+$=hJQWL~=~85d}byY8dtuF=gk{GBIJ<Do9|a-Ej~'
            'bN=r?#23BupL^%(7!Ud<y|c>!mq96uvVW<$%d4D9xqGENiT_pI+?TpHS!VMJiQ*^$H_A)f*MQA0{_(C6{=XWQWhJOq=&<up{SWj$'
            '4SN04pS#EXBM&g|{pEn5SD`5VMLavG`BJHEEv)p74JeV@+>cPzT{^zyby%8Lvsw|;Y^0KO({;~IA$OR(FSOG>OToW3p+(907xWT$'
            '=aYs1O2J<iF#SBUy|scuwvX)ufA{~m(WRz(P1$UliwpA4tz`wLjI`LJ^su65>1gBCN9nk4vhHlD5U6I-lz&ZyEl_K&Y-JhtiGBI^'
            'l?^L0E85viEP<wYQ^D+&3=eg9fPC^<<!JV;1}#V$Aki_Y<ehK4D^nhQH{TIgzI0;<qfPjEb<lctt?c#w72?|U^-i5MP`qIwvE;Ri'
            'yr~Ife(<k$X)kxLA3-qk$^N#RBQD5IPCi$)gtDGZR~B(J4rU?a26}yJ%RZXxot>xTTUg%Cb~38t?0gOR>PyH~HS*LaQ=#9lx!B~V'
            '%xnx|WHY8hCgU0k^_djuQ>oEo`AXsUY;%pCRECaud0elCZKFm_oouzRuC-+0%G6InOeWEIFzC~62}5?TL3aV=CRhYk{Gl@rXIUsN'
            'e?7W9zdSza9`)Ofkp%1aAaenUzj!j9+_kGEy-%dsl;YROx9bBtLrLe7i{451bkIM%_$tYH0uEK<x_+FGi~Q9nrp(Buf%TMuq@{+h'
            '*1unzoyrtmdw!DmYx16orxeblg4?7USQ;;=P(~{56Q4as!i>pkoKi%$FNVV*&O%JFVyxR1=V+%+MWrvCs<0?atgt_exLKj^nBfde'
            'm%VZ|Iz^-WyPTv^8pf->*KkI0$`x6q$F>YlDvrjtbyV{fJ<;3jxZOw`SY6|X16-i{!*A)gjrMmxxbO@J8z+l`wio)6ng%E(n4+g`'
            '^dw(4p1IZ+RpBXgKyHM@wL(%{X>L<E)D7nW1HX*FuN;@m_94bgw4S+nmNWQ6JKylY*kDUQ5s)&U(PK1^Z&rAdEuj{$pznX+i6uOf'
            'M?Ew7;2Nj)ofJ&a(;d3~rBkewwQ(B8=V$)Xj~4zSOx51m*V=U8jWVt{lPmrfeE8bBv#!6Z*fg6dPsd*;WEYZR0SFy10xX_GM2+w+'
            'eUqYGJJSYBlg&Izoinm^8UVWSZ7`nUPuEDx#~lsoBD!w|7y?C65qpx-<gcYm=Y{26${uqxdAtY2G|i^7DZM@r!v_bWkjXd3aUO>('
            '`ld4ZoeU9y{;>A>9Fw%o@tlew&wI$#)>FKk#B&o*sbeDob+U<a*Mz*+Bm`Xc2#n`0d<E!GDbG?NVVi<mAvxgY-BM|cc;!)$!vBR$'
            'f%TEmnvD@EYC3Jt!-;y{qeLz%ZC+uwtk4r~Bm+)(w5s@$x;e^3u)?2XLabRDV&pZA`G@4u2rIe6_!O;<Tv3Z_a~|BlNUmWP{H2$i'
            'R*7~Ir%TTi`Efsc&OlMiGe+nYHXgyln6jZ9b3B73B3d*`2vcA0xAMMm^9uY3&Mw}`&d~Kao{QZFobKZ;S%L-dr0R2B$lX%PbpL4|'
            ')dCcJiau|L<V(#{M*0wIm*<>rd7maACJm}{UoTP63ZhjdU6e0{^%HkH%UWRtYWfxk)MUPb1lLWn5M7(dK8DMaNfgDi{jZcj@z#I-'
            '{VN8S&Wx$^4Lp^}5~g-L4@Yetr7^-V>D%DLB)kD-qTz1LbJ?Jnq0u@EQunhGUSj^dB*P*uDVfHqlDBP0sYx;cRZtHas($)T{n`jP'
            '*5WJxVrlUPRkpSO$c^D^h0*v+6h1;uHC*Z1Nk(w6fpFXcIh+S}VgyPufb=Q^wXDGjWDjIZstMnC_5JrRUcrgOZLmoIPeX<8(MWcb'
            '7r!sBPKJWZ`yPp$4dr%=YK%&+@@-RCp<qk$JIpxg`=+DE=1Lh}T~+Z$`kkvLPlkib<_zh|0a>lwzPgT(VsNgDRt~FEeR%0=gwfT&'
            'bnSTbjbNa#Y43Ig1kUr9sIOw45;OQrMrE40lI6!YMKk#&nu6%W0XeKbw^WpS9@@hpIG}o=6<<d<DJ6Q3Fkp}D1bc|uduT#z$y_$l'
            'MGaXxN5*q_52Z53-2AKTHcrBiEI8f*=AxzBH#}xw9+&6GO;+8|&neJcl?goyaTKJJr{!U<EEFB0o0KEg{W%_twYW#0QW?I*QBakh'
            'J4074oGJ-SR}uUFa}@v)VBx)8-Wg2p0v|)G!h{2<J^Iq_YYNnkd1+*UVo{amsshM{<R|DY!)1WEirQ+o<qzYexx+&8ALZsHgEKlR'
            'N&`i!l6Q#C+a%Ab={l_3|7On9Y>^zUHVlMDN@>1vji_#Grkod40#BB-wm`QrN2qP<s-#NB*6fw{MF8{2f<^H0HEacDPXKkPlynLZ'
            'xs~}N5Q#zoI@Ds~i$?9<xG0dJYjw$CTF=M|jxY?5Wl-vu49xm4iE>wdV}>1oV!4dNC?mt7qQl1f{9#>->OhW~n2hq2J~u3*ZY0%g'
            '+X#)iaXv9rqh_pf(~9vbB0DM$P_dD_A@YM1kn$LIGuPc18?vFhiX^CtyeORIO#v9@-xSPVLpADyGJp;E3=`)dFM^8|JgjoG)zv%-'
            'ktF4>o9~+#Y2t2*dMWoWCTZFxCZ;?}O;dM^$eE`H<Cf-FmWU+<Bit-Q_P3g_>CSvoj<l-`Gb(?>62_<woBg0TtWD|78QNcPjj=Y)'
            '(JjP06$CVWu*$Ejqm%@oQ-<iC1l#VxXTmXV?VIyn&l~i6=iX6w(8V%S{llKouR8h>O<CH_X~UkYvNV(?brReJ30-k;ahgH*r<4Ba'
            'QUAQDs?w~3YzCA(l^KpF$eL6{lTwYUrOe!kU9zq#$8N!yTx@o!nYp?q%O7!-9(fn9ANu7mG+list!Q6DCcE!rO|BBPhWczJs>N4O'
            ';NJ&HD`%-FI>r$iG(q@Y06-_HR_L3k$StH|4?TN^KyGklLFI!m<-pl5r)0jG$YM*x<J%ys25XFU#>0{XD67L@<Ss*`B5BI7^n~ni'
            '%GZVhH?Y<>W=~1j*@nE6xK0hq?us?Qa1`IaSk413jQ3S65p>Eu6ggzlA<lS2nq_UWu&wfrZL~_&CLGJt>U64=w{O2?0DH50Ke<Vz'
            '_$mv0c514R@S<F!GPM1^yizwWzlMhKx=a#Qt{yx#I0yckX&~G_(N(7+(N$sisq;`C@A0?jx9EYt{8-GYgyAOD>E&v|sd<IPZ7%$6'
            'Z%qZVlpj@Xd+CWiZjKN7SF<uyk~1RFHnnTg$no7}Ctsn0YDfKON{Ds%ABhFZ^oTd%PZ(cq(`CV>rm-pM7K&~5Sc2?N)uS=nmIaNh'
            'z@3AtARTEkw~mfgf~?w+4ev6IctBNstS%!FCnK`6<EY=w(#ciz%T|#Z={C+5@q#@kP^XsXRwX?AR(9Mbhs-pLfc(RNEgn(uh@!!z'
            'bGpq!+O-afA*+QlMVtrYAiN7wA!7@H5G3%o^lPP?G0cKBiSJXfC(<z>8W0z(+7iQGAXTR7J{viM3LHZPCKin}fC(HQpZ(&UcYpCP'
            'tfK1;&O9yE6ms}g>{bC$FqN6Z>xAEOuPeGSGGXhTJn5zEhl%G=8(PC_iB`)4>)K5zzWf;=;i#$c9TK{HEU)ytYZslyq$KBBxMT1e'
            'r?}mY!F|51MG}U-qGXuW;`d^S+|PVOjU{>F-YqvlUYdUD9>_H<!U(W2y2);>O+e^k(p0$v>Km9IRr?IMSUdLR(=abVZ#$rmR`Z!B'
            'gY{(ozTW}%+wkVLbr;W9i@?m?f@z2p=zlC9&@ciDHCGHjK@Wl?z8Chapj(*DgFVioP1Z{?m1h2pvB6E_9CMDRdEiH@B~VeRV`^Vx'
            '+1{8Y-5>*hxI?eqIQm2sso$Ck7;m>ZcVXfK<p&OyER!Jny}JVs4$u1odT0-du0Z4#j(?`)G9c*VW^NL-fi6{eRZcx5^`@oXm{pIA'
            'vAL<W1edDiZ3h^Z7^uLo^xCnZ!s1Atxme{r`~DLlR+Si17P6)%5+sL;yIi6dl+0K9y#2kAN|N*d031pq6Pc>X7Z)eME=06ii~yFn'
            '2k}y7CQ~!lIqb14uD@m}CRw1GK?!Mn>=TYD-4<7ir^I41ZPUG!xkjsuoVeI8#5d}6(L{cYSA*3kz%!DBV-io~$_IEFQ_A&&=^#Uu'
            'nCOBFHeQ||r+ed&w=N~UM&2U_d|@%eOME#Utw=t_1U1wE8S<M*ogv3Wp%Nlg;*aTQs0a7Bt|Yrls@w0<+lFQKG=0I{x^e%xB0$xQ'
            'E8lKCu8qj-)owkqS}QQX`@29^fY14n^HyQ7pI}7XMfa$8*b}=WEWsPC04?xB>DC{SWn=;f5$@ux@Cq#=IvZ+Yxp!K)Gb4k_U9k2G'
            '2=oy?7ZC<w$Z90k@C%Eu^$t&Ze=j(M+D@IO=VQh0m@Wx;qUXAts&<1+4O}H2k}B`VVGhNs>vlf6;uR`8^&58o@zE)A8e<+bLg*WP'
            '+vkU3_*Qj4QN6_;UykWFhj>%`OE4M=bW2J_W0-%FlitHL?t@W#m4the-Ng$(jD}KjR9Wb+uJbId+JSRS*Nf=)W6&M7l#=h23AtK_'
            '1X+hbLK~Q65Ja+)6c(j&H|LPIoKZIBk=M*wr^YuE$d@<9DHU~7Zw@wB{;_WU<)?IQUh_-59o3}}V26x}VngT(4-175U=#CZ@EU6p'
            'Bw@K)0!|;>!k6$LOlb^YK6tvu52sZNqP9PoG@z?JvfQ){(L*?!1N>%HG!9}PytSs!4=X%>97fY(d->F@jM|YXCZ}0@(lx!@c+#<='
            '%n&{*voEV&L!&YSC;vq5n5HXqGpyoG65Y{3S&>Q*+`<LbkQ`C6!m&&hi75anBj>#jG=@GK<V9=c1o+nL3Z#<jEy{Xqe_(J3{KR^5'
            'ZI7yx!?;=iR5hXla`H!{=c8IwY5(O5g=^Hw+GW{5S%}48aG`{!BliF$q2Bv0Mw?1iTU$q^l#Ua?eIjQg@|jD7G<Lr2o%B!p9*pSy'
            'eAYkgU1*l){5@FSt`jQzuo}<s-@{u!AtBRwhc`s<@0%b>LTYmyOnm(J4EP=X+l>QVe$hDu)CrU1Hl|M9RTfA5vj-~l+@EllzP}7I'
            '{&~3=!HVb9dVUiIBVP@51X~R1DRl-e2#T_76hz^O3Q_o}4}XE0MAP!2#Uu!+Ll=-NiTp&3)tyt|627`X{PSrgZ~bwy`oQa+V~+$B'
            'EZIFw$X(#?z_X(OzquNPYP6p*?e!w`fs<433qYCi?T9~(f6PJK3g{CTtGPe*vBtH#qY^UfEx`B_g0J=JxO@8cvip|aW=U&s?(BSX'
            'Adcekip}6ka7Nr<yWxR&{!+|-`q&iDUbRLcS!sd0p~dC>;fZL#?=hEw3KHOu_>qh5$$Kn$iKx_p?%w|XzWL&p{?Xtkd_xZpf9jr|'
            '_Kp!{o)sp8vmoQ)2W1fA4;-GI4tjrs6`j57opNk6^nUT2J|s==ii!~~d*qX67nsRbfZkX1qG`B3$>aB5sq9Y}p%z!F`@@gD0xvfG'
            '6RE+LOw4Be+!fhJd33BgP|-3kMo^keStuyGH8S_N-5M+AhdS-6m+jH3my>|}9PxQoIwB%@yzSJaN%P)$G7tiBj$$$Z-+TpnCWv<='
            'I?M13MziI3;bpOR@83xjg9isfA(f95)I%eAr*JONl8p`^^R#20ff{c*InaGL$!<HvgTx;G_(jc$e@Xmhs*fn3pfAq*Z{F<v(*3!Y'
            'pH?YY8z)|d2T2Duj5@#pxTv!jwqcb+OnM`Wx{Y81P#Ry({WaeEvZIhpFiqhL!4kCX{Em&CL)qAagd+oS0aT=+iyr{h7)?~DY)j`l'
            'S`(<v(<xO&z|7Mzm5!k~*upfnB|uXKD-Cw>t%fuFG{Kk`gYx`&c!Lu4rj`s5#6Jfi>4@*-I@(JRLmJLs`aqChyrQDt5h4Kv<cu7I'
            '_;is8?2uRLEOh<pr#}3@gA}%jm#AtY76vu*HCAJZv6!Z{09enTT<iNIKGv0s(DSz(hk5zmA-)%5ZkWJoXJNMdgb|;I&a?gJFVV?!'
            '$zDMljzonqNMY(8p7ilv8h!+2Y{gc_y(o@aXj6v<fTv{nl@=ZLlmmPN=u0GjPZ2gGzTp0;1%+7}%%@F>e^C%jsJ6{?$_W6`dcsI4'
            'MO3=5knwDkem%?zh)5aIJMg{imVKvXQ8~0rQ{|Z80|=w&8h~%xVakw04xsM{a_0kmR*}NgWgT(|Iv$du^w}%Y7*u})+@2Bq3op=5'
            'v_6rMqoC|@p!%I2^ikg2!nEeRAY2EzNhQ2sH9GA-WrH+^V;`o<D{XVKM%Dv)7V;+`gQVUPQ;bl1`Qp)Z%x7?4DOg{ZbZVS+np%Dy'
            '^6KC~cE(BbBGF}we^uyQc|}n~uSuM)F}+bdU!@etE}_M&i#8lxHzliZO;BE$KvRhba^9v^q;bS*BK|5~zO>q@{Vg=><R>#Pdt|(Q'
            '`I2xR|GqkSd5st5is0$ze>!;fT)si-W?ZGKkK0&#EnU5HXBH+H<mgsQ>^%BE+-ojshg516NjjwoCQa!1{=th^*SQQp?o0@Pik^Md'
            '>E-^x%a_;g)&8|nm4<dg`Pe)Z5Yn2SiI;i>^(m{$IEzn=fb|@7_E9F=p$RYq{4?4Nh7J7nsys~L+|wu~TkLcyNwQ_pG22tx9K{~h'
            '#|!Zo@`%Hy@(tJ%Tr#MZB{ygldwam4Y#~k`3yXB*QR(V&DXW44L79SKXFW@mlY{91#`U8g3iWMMz+PIoOfpCY|M(o5;S?$m%dc~}'
            'Sf(H{Q(79DQEPP$?cWL1137I6bH7<r+~pB~ZS=>2l_<h^=c?+$>2i))bNRv&I(k`_Hsqf=m|b4sfWEG8Z(f3mRW|)x;Ap8hJ3aoD'
            'jiTyqt}VLi2AHB3rq~Q?{ZR;O7?EQbTkqJVOP0!8EGS4J4OFca!^exXLfEci-B)w^s=%cg^eu@UhlKs5W2~)Ya>X{+I#i6dN7{|p'
            'CfQUMJ<<^C<dr;A6m>B!m%>@w1m{NLG=BI?=c@gOAFkcs((hyxyL(_)LysDAR9DK8YAg!w29(t^<^#!%3)WVvzN>7a@3HNsdI{<~'
            'JEjB7K}};!p~rtbV(DV;`REiuUHG)gC7zt2@;9QE<(tQ9RE<}erv=cvI2`MOS|!^^C8eJG(aj1yp~prot}5F}W$y@6IuN~?go&!s'
            'E$ewX#yKJ`KnpWZROOIGm!7RIOLJvToAy|D!h;vzN0qic6(Y1sW?_1Zb>dVB&c@IJOJ|piM!Q2-`Emh0#R@wFZ(XZX14C$FV15dz'
            'j&f3qrK$w!%cyw}CRn+o40k|8Ebp+1XlStlzYT0sNeqHnGBTW|$oYb$AE?XW?)&%WXFqq34^*&N(cFd88y13&hN$ZLD<A?RtPBN>'
            'cXUY`zHohi-ut=V`vn$Bckv-ty@LkfbS>+VD!@l-MbJ}vB6H~1P>1CZ?bd$ELpTmC_K;HPL}1P6cHuU;LT~QYN)W@X*kgM;TFlu='
            'AdGHdh|DhiL`|`n%WggyyysSQSV#SE_#4vfDON>mVyPnO(kzCyTlJ6?QLzLWSlxi{Xt9sxZEH;G+$ya}{q0(BK}97xv6ul$HTaKI'
            'rH?}~-;>$otO(r=ETKXtkRd`&mZdV9t0JvZUb~}QQ=vjZC`OY|Vica$YstxO*s7$8RaA{qQ+1^_ereFXc;}s;T@HE|SG1h#stcoH'
            'z!HsU7gwpMV<Jn5=sM|7NfMRBk9SE{$p(C!h98iFS5Q&4UnuLK8?M8`%!Y<6O*z}RE*sO#3n(!ylh-Y!n~e<-eNzYh+oMIW1_{WZ'
            'EFjonk^>s|56L44y|a()^2n|u=P-`v97H8=u1H^#cUPb6I4y^2qZ5`{>f5@7<;^VBEZ~dj&Qp%R<!thGp1}5oe<wx1Dg!mT30Eg;'
            'Hp<_|6QHFTCkDH6x|c`&Gv~Tm4RZi60s3Tc72gA~ArrL4^6*HRe{az8XoHO}z_AU`Gd%N3Z}<UvGr*tp$Kzv7*5~r$$4yBrjr$Hz'
            'p_Q$E<84UQ-GR~YV5g=eLcIZ<h#ashX)&L~5%A^7$|+U?QOGHULF1!U<y{yZ#W%1aX;$>PNM#~Zb;-K(8JRhh`AE%6<-i&_QpeJ7'
            'J|T^ZBT0Y)2EfKCt*+t=J731BTsmx3x|K^#_i29Rg+eGg-`yWL^txP!h<BCDgxAFiBX*lSyHZulQno~W>w?C$;zdj`!)QUAxCdcK'
            'n%f$>lWU5cY|E%JbK4S1%`(v*x~XAdt@)o;TOtkI0Dl$I^_gUfEPdt^t8E3pq#hN|5$QGLRe$M<qs)o?r@bt8B2j{X3~(bz7!i*a'
            '!!0L*l~68s-T?Qdl-Bc%v)bCIAueT8kFz<HJo0i@ohMzCBK>Q@xf9XPlO5;^%0XLP00J7$co?Y2S<AvC4fa450Ld+)V10fhWb#uA'
            '+K;ms4skdWH^CTPo<G7q0HrVoTnCe2h7?2!^+tEBAo25g_xPl1JMbxUk4CC1WxcxITqmqpIG{djr8%~GT<bi@@lil+@-z_mhn;Sv'
            'tkQq0ADZb+Dr$z*GUbXGSPp$FYtuS%U28?HF-R-0L{xCKD#TYz=B4RM>CfHyJsP&k>SIIVD5DqM@edi&!@XfijTknyBJKZw-Cb>O'
            '8#faE?q5NuA1oD;6Zeo@YeNN`T2726PGKj9GYmtbv|D+hH&?55qqzR>H*e&UORg;YQXFvk&?YOA%N@QAXP$Y6TciuT#>m`@caLAa'
            '#9zDVQpn0J-w1ViAG9}6|AK~GVdZdbVBBz6BH}DNC1H_mjyZO^_9!wzqTNQIXGRpi?$s`-eY@8!XA*dE-d*=DvJ++0=~llkwi`J&'
            'DE|0vl$->l#o=SM5_73Vs%^>&>sHu7oarX}l4LzR#C-vFWY{%8(c?WPLG-UWJ-awMga7{0d*A!eyEs*EdcVFuz5IAyO^?y|XvssK'
            'M-bwEb^N0vOboGw8F|*;2{}ltgT{$TFVEhCpbyTEZu~o|;8_x>!Kbdz!N4ag6b=T~KZj|2V`jsa*n%6K2D+Uhjhp+%BQJ1j#*X!{'
            'eyo&7iV-`x!)BtzoOZrLgc#oQ73jh95gF~6dkakfs}ZGRhDP^zZwU8WJI&lSVJ}1YI{aF|+|GB^Zdo3UNquTM&81(vYK%@M_S!=)'
            'Za{VptNeHV{%xDutCI%4A~xu8VTg_C>R|u%wb7&M1#K)z-oHKW4v<cg-r1bUeM=i!&6!(qus56WMc^o%)Ytom7XNJgR+Xo2tL}^$'
            'E%H^r$dkUggaj#WZ3%|_hWT;-C~ym3_bH&Wzu$iq776-Xx3=BSa{ItaYgo$w%RRCjSBtFXX$UmPNKYyye6kWcYV*E1X-g6cZzvx&'
            'QaY&KY?vkqQGtnZA3-zEVka;gRPVzZ%yoA5wtsy3p?7w1`tb}7?|;7jF;apuqbd%ehsD!LO?Ns!G<FcV89qmIF9K=?c*YYraB#>N'
            'V)(ZjFH(!FVapIR819tsBts?~c_*j;CQvc&`3*yv+0V0CHq}e!eQF3>pW*T?e^h@dRx1t4b~MGD@JOpx)+?U{#hFS1M>i34s{1+$'
            'LzA2KU&@q-)h1AOeO3CI>OF*BFDq82kIXGUp_tQ%i{5I;F$rzr>VuO%o#*7WVz<<W8M2vD5SKHa<fMdE$O50`(<(n)-DR3)3w5eP'
            'FyOOK%8-LGrqHO39Yd=l+Br<>@+))C$))X}QD7mupE*JheM+r8H3(<+KztgjMaL%2$-RTmyouCtSOYK~u;t-l<_5nKne#0u25EB0'
            '7HcOG!unlkS8A7y^1v&xqoNb&7OpEKadq+Tr~`+Q1S{!3SsS*br!+tbBRI)bKd1~Dc~L4HeFP&w12oON>j)!hE~Lk#1(FYfp89;L'
            '*vbaKOF7VgpZjIJm=&ORz=awa_U8oVZ9ksY{xQNh&*9>H$91b6^|OQxZ{c^5)G+KTso5jxFskkf6wPj`qwfp>FIUzIkESXdn-&w{'
            'BZUuhvd;1+I~<}^AtaEvyI_N>fP32BiFpRh4b83yE!=;w4K(e;)`QDt@(hEpv-NaqB7~g54g%uJ-=1JD_JpNu9)*|@{Vv6f@p&25'
            'WE?xpriAIvYkI&Cr}Cqy0(=cWkB1ZRZ%_0F*bYEvC(zlVV*hr)FWi0$4axo}KJAq?>5D8!{cy2{?eN|<{~$#5>fY%@B;TXoXA!;e'
            'TwF2E?T(iaA+Wfm>sA%xBQ@ze32ZGB{Ru9Gok(^juc-$$&X!5W32E=$w?4=?@N{a_IFbx2I9GbECt@=TYhKKnaj)!(r6>tc<de4Y'
            'XLu*bliV7vvC~csGIaWL(JNTpXQ_PkVO<T*5k`OYim)i~-YH1nE05H$Zl4aDAeugFvZrEBo)TY83k<&WnBPw|YfZaw(ZeXDQq64Y'
            'wi=RRM{DB1)c~~hsNr9*TiULJ;U==&M(`S?*KmO^zG8xe$x72kxtPv0Y?d;`F|Qb{`lakQ$%4RcFz>#o@70>5_@%9Aqx_oPa>uzS'
            '>S%rKl>Wk{DVU}O(+5uES7phelw8|(snMZVl4QWmcGXLk1lR4lN#I1taN>J(=Fs;SsNhlq-%C{!hBRN?rBid6E<?=vynx%*!rQl-'
            'k3T%1=k2Qw0yShj-Mf<>YgF}YBcM3m7DG^bZqnpMNwX~{mT24idoL-c<ch35(i+FEGczbLl_KtT&frd!6rcs8^1P>xQ&^g)aU+ol'
            'H8;-gQYUMs*?nWlOsd}+NOoT&J7gMr%&rqo;!uVQ<oe*HM}Kv*+*!GL%Y=Z+*gTp>DA&+F{sl$Tp|`g{JA7@%5L5ZsTo8RLjeNs!'
            'B8?B?u@+Of_|b5}_q^Qu>4y-Zdu-Z6V6r&+^k})ra|9u;#0F)^8DtO#=0#!78Kg%W;+=&1g!!i=c#NXG*(Rqnp|120rYARO`#V5='
            '3*Rd<gK}i~<7pc4Vz#}vx<gP7Gc5u{t!y!<(t8j0PSt<zI<oaOR_>OK>2FY%2qr>ZYj`#Zo&?k;j&(Ke9&@QRrA3T}Uw4I>YIdMH'
            'vn#i)Yco;APSGE(-EX6}PtlypQ?X`l{P-%I8*i6snylM2GwiYxswZZ)aZDpkMs0Vd8G^R$J31JcnUs`XQo{t_;~oF{k&&!^K`D0P'
            'H%g2Z!G|#MdkDm2URhNx!--&qU#eU+MyLQUvV_>xYKPwWu&c%lm5rs7m67@%hN4O&3c1aLx~=;5%PyhZqPkRbgdr64f{xlph#V-O'
            'FZhil90>-%!!6QoIQ`9^Fm<RM0xy)uPc~e+FL0F#j9kX3Ts=k<a4jYQA_mcRQEu1LO@HxRxpha5Odt5$-6?S`OULf2eR#Wu6SFBN'
            'rz{c1D#~x}viZlW$dUOQmoxSo#b9P6)ar0nCE8_qKcPfTsa|$$7wmy)mW*^ojwJug;Om8yXtg5eOp}ilQv@c4^C@!7uo?LX4w&&$'
            '-{ILVcI1=R0R;ptGs1+!vix{*;+n|nId0KHsmm5E2~^*0FQ5j4ZXr|j_PIoYKa}r$0d32zf$f%Cr#|^6XnXe)J+WGHGPu3K9xun~'
            'vG8O20Q>XLXMN0A`S$$iU#Fig`{%uj-qHCl?~GIi%=I>dnM}-KAaM9pPO?WBc9U~M?nKN4i=`R)ExMZbjNM{B0A1WqZZS#^qEQX-'
            'Q%?-6g|cmO)nU(8OmnYvzEegyyQw?LyFUd5?+BGXTnEtppgNKRH7t^Giq=YM2W5Tq>ErpOgy{-?A<w4Fv<B-aN3ugyJQBU$%>@A`'
            'Qa@eg4z^zq6$5wZ2C#7kppB?*X{K#p!>lB(6@zAMUDTU4jq52wCtg}T-F`QsrAlLGvr#v<7z3l4cY2xYHJpm`)&V7sUFXQsv(yM+'
            'nrZb|A{9*eD_|#tuD8`qHX3G22DO)v_Pr|o(0>PLkBYU5aDOJcz0yc^Zel*=%A*K-JCR{kiYZcZR?X^j3PpRlb)J~5wp+IQ=utLN'
            'I2u7$hQ(@>=0UO;NjI1jOQg}wL5%=ae^{)G39l%%Dkhk^0!KF2lU9%=M%HaN{-KuJej|(c&s}Nef4Im5o%7i8`n_SimimU^1S}<7'
            'Q6Svv_jm94yB;22RWd%%qRzARVtM#~y`Weu?lSroK8MXw3Xeo6)HodiHg(<(n)_`t^*ftqcf=v_41o9MS8QP?b#M*w#Y3i3Hzley'
            'p>9oz9@tnT3CDxUaIMVhujEWOnuHO*XcAPRycEV@>D@Y5x(j>tPSq{Iua!;txOdjOyy(4G+SLIysD8y^h!_5hD7_h~P%zSnEjFo5'
            '+SLu25W=t>7c=JmWMJ_u{b~=|JYB=Cu+A{!ss0*l$n{uUci_t;hz=gNq5y9X?et-twG(Uu&eJ=}^ca2kjR3i6(48S{6X~=JLbFC)'
            '^1+sm0v3Jz;Z=b9^g!oju^=6sKU>R!(RJV@M5`94_GsG`^7m(pIa#J@2h$2M?TO@192a_-m^?8*efzwx#O#e|hYyG!{d|j=$=@It'
            'I!&7A#T>blNt=>W1HtgJ91s@2T=h%rsYY&Ds7C)UpYv4}jbxVl+=6W;dOg<@&Op$hg;O%Q!VTg){k<Uep#?_TCq&FOTHLOe+4u(U'
            'OVmF8O3}R_KSlBbT@0Cx@F7%RN?LWkH6azk?TSLE<(l&zsfpCd$?P^ABkzVr!$;@On+y)iB|P|TeSwJ-j>-B9|DI;U<syfjJVky`'
            '2`pabyz%O}p00C3i@^FjWVA?qmWDQwQ$yL=E#g4%Jue8Ew`1+GLRnxL^(@=$c!6Kec+BXXt2z!9O*ua1JHB^xQ$h&73mvF?XJj8#'
            'F@mK68nBw^)eOW@-I^~y|N6dnhQ!Il?G;U$g(0izvY23`{Y2s34Z2*8B&K@3)Z;+^MumN(L2pO=_BdUplWC@I^kfPbGSDh-vRk#7'
            'CdC*JgH^hQt8lvVwtTV3*r^QX?mU~1n|=ac*#jIc;Pd6t`RT#Nv>Aj7+EefJjHmevhh^Q$$_ZLwv1M}{U_|d`1iE=lSZ@lrx<Fx&'
            '4Oc_KIjz#YW}{>h^d~{D;NLbP=0BPWT9N'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
render_utils = _local_pipeline_modules['render_utils']
google_ai = _local_pipeline_modules['google_ai']



## 1. Test Videosu

İki seçenek: (A) kendi dosyanı yükle, (B) bir YouTube URL'sinden kısa bir klip indir. Gerçekçi bir karşılaştırma için **3–6 dakikalık** bir örnek yeterli (tüm videoyu değil).

**YouTube "Sign in to confirm you're not a bot" hatası alırsan** (Colab IP'leri sık engellenir), iki yoldan biri:
- **En garantili — SECENEK A (upload):** Videoyu kendi bilgisayarına indir (ör. tarayıcı uzantısıyla ya da yerelde `yt-dlp`), Colab sol paneldeki 📁 **Files**'tan yükle, `SOURCE_MODE = 'upload'` yap ve `UPLOADED_PATH`'i dosya adına eşitle.
- **YouTube'dan indirmeye devam etmek istersen — cookies:** Chrome/Firefox'a "Get cookies.txt LOCALLY" gibi bir uzantı kur, YouTube'da oturum açıkken `cookies.txt` dışa aktar, Colab Files'a yükle ve aşağıda `COOKIES_FILE = 'cookies.txt'` yaz. Bu, yt-dlp'ye "gerçek bir tarayıcı oturumu" gösterip bot kontrolünü çoğu zaman aşar.

In [ ]:
# SECENEK A: Kendi dosyani kullan. Colab ve yerel Jupyter otomatik algilanir.
# SECENEK B: YouTube URL'sinden indir.

SOURCE_MODE = 'upload'  # 'upload' veya 'youtube'
YOUTUBE_URL = ''
VIDEO_FILENAME = 'THE ABSOLUTE BEST OF DAEMON TARGARYEN - ABC CINEMA (1080p).mp4'
COOKIES_FILE = ''
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240

if IS_COLAB:
    SEARCH_ROOTS = [Path('/content/sample_data'), Path('/content')]
else:
    SEARCH_ROOTS = [
        PROJECT_ROOT / 'sample_data',
        PROJECT_ROOT,
        Path.home() / 'Downloads',
        Path.home() / 'Videos',
    ]

UPLOADED_PATH = str(SEARCH_ROOTS[0] / VIDEO_FILENAME)
RAW_VIDEO = str(RUNTIME_DIR / 'source_raw.mp4')
VIDEO_PATH = str(RUNTIME_DIR / 'source_clip.mp4')
AUDIO_PATH = str(RUNTIME_DIR / 'source_clip.wav')

def resolve_uploaded_path(path):
    """Tam yol yoksa Colab veya yerel arama klasorlerinde kaynak videoyu bulur."""
    requested = Path(path)
    if requested.is_file():
        return str(requested.resolve())

    print(f'UYARI: tam yol bulunamadi -> {requested}')
    generated_names = {'source_clip.mp4', 'input_video.mp4'}
    exact_matches = []
    fallback_matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        exact = root / VIDEO_FILENAME
        if exact.is_file():
            exact_matches.append(exact)
        for extension in ('*.mp4', '*.mkv', '*.webm', '*.mov'):
            fallback_matches.extend(
                item for item in root.rglob(extension)
                if item.name not in generated_names
            )

    candidates = exact_matches or sorted(set(fallback_matches))
    print('Bulunan kaynak videolar:', [str(item) for item in candidates])
    if len(candidates) == 1:
        selected = str(candidates[0].resolve())
        print('Otomatik kullanilan video:', selected)
        return selected
    if len(candidates) > 1:
        raise FileNotFoundError(
            f'Birden fazla video bulundu. VIDEO_FILENAME veya UPLOADED_PATH degerini netlestir: {candidates}'
        )
    raise FileNotFoundError(
        f'Video bulunamadi. Aranan klasorler: {[str(root) for root in SEARCH_ROOTS]}'
    )

def download_youtube(url, out_path, cookies_file=''):
    base_cmd = [
        'yt-dlp',
        '-f', 'bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b',
        '--merge-output-format', 'mp4',
        '--no-playlist',
        '-o', out_path,
    ]
    attempts = [['--cookies', cookies_file]] if cookies_file else []
    attempts += [[], ['--extractor-args', 'youtube:player_client=android']]
    last_stderr = ''
    for extra in attempts:
        proc = subprocess.run(base_cmd + extra + [url], capture_output=True, text=True)
        if proc.returncode == 0:
            print('Indirme basarili.')
            return
        last_stderr = proc.stderr
    print(last_stderr[-3000:])
    raise RuntimeError('yt-dlp indirme basarisiz oldu; upload modunu veya cookies dosyasini kullan.')

if SOURCE_MODE == 'youtube':
    assert YOUTUBE_URL, 'YOUTUBE_URL bos olamaz'
    download_youtube(YOUTUBE_URL, RAW_VIDEO, cookies_file=COOKIES_FILE)
else:
    RAW_VIDEO = resolve_uploaded_path(UPLOADED_PATH)

file_size_mb = os.path.getsize(RAW_VIDEO) / (1024 ** 2)
print(f'Kaynak video: {RAW_VIDEO}')
print(f'Dosya boyutu: {file_size_mb:.2f} MB')
if file_size_mb < 1:
    raise RuntimeError('Video dosyasi cok kucuk; yukleme veya kopyalama tamamlanmamis olabilir.')

if FFPROBE_BIN:
    probe = subprocess.run(
        [
            FFPROBE_BIN, '-v', 'error',
            '-show_entries', 'stream=index,codec_type,codec_name,width,height,duration',
            '-of', 'json', RAW_VIDEO,
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(f'FFprobe kaynak videoyu okuyamadi: {probe.stderr[-3000:]}')
    streams = json.loads(probe.stdout).get('streams', [])
    print('Bulunan akislar:', streams)
    if not any(stream.get('codec_type') == 'video' for stream in streams):
        raise RuntimeError('Dosyada video akisi bulunamadi.')
    if not any(stream.get('codec_type') == 'audio' for stream in streams):
        raise RuntimeError('Dosyada ses akisi bulunamadi; transcript uretilemez.')
else:
    print('FFprobe yok; akis kontrolu FFmpeg donusumu sirasinda yapilacak.')

# Kanonik klip+ses cikarimi (master notebook ve verify_local.py ile ayni fonksiyon).
video_analysis.extract_clip_and_audio(
    FFMPEG_BIN, RAW_VIDEO, VIDEO_PATH, AUDIO_PATH, CLIP_START_SEC, CLIP_DURATION_SEC,
)

print('Klip hazir:', VIDEO_PATH)
print('Ses hazir:', AUDIO_PATH)

## 2. Transcript Çıkarımı — Aday Modeller

Adaylar: `faster-whisper` (tiny / small / large-v3-turbo, lokal & ücretsiz) ve `OpenAI gpt-4o-mini-transcribe` (API, ücretli — anahtar varsa).

In [ ]:
import gc

transcripts = {}
transcript_segments = {}

try:
    cuda_check = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    cuda_available = cuda_check.returncode == 0
except FileNotFoundError:
    cuda_available = False
local_gpu_opt_in = os.environ.get('YOUTUBE_AUTOMATION_USE_GPU') == '1'
cuda_available = cuda_available and (IS_COLAB or local_gpu_opt_in)
TRANSCRIBE_DEVICE = 'cuda' if cuda_available else 'cpu'
TRANSCRIBE_COMPUTE_TYPE = 'float16' if TRANSCRIBE_DEVICE == 'cuda' else 'int8'
# Karar (gercek test sonuclarina gore): large-v3-turbo GPU'da varsayilan
# (~%91 ortalama kelime guveni), small CPU fallback (~%85). tiny (~%72) sadece
# karsilastirma icin tutuluyor, production icin onerilmez.
TRANSCRIPT_MODELS = ['tiny', 'small']
if TRANSCRIBE_DEVICE == 'cuda':
    TRANSCRIPT_MODELS.append('large-v3-turbo')
else:
    print('GPU bulunamadi; large-v3-turbo cok yavas olacagi icin atlandi. CPU fallback: small.')

print('Transkripsiyon cihazi:', TRANSCRIBE_DEVICE, '| modeller:', TRANSCRIPT_MODELS)

# Bu notebook'un katkisi: AYNI kanonik transcribe_audio() fonksiyonunu birden
# fazla model boyutuyla calistirip karsilastirmak.
for size in TRANSCRIPT_MODELS:
    candidate_name = f'faster-whisper-{size}'
    t0 = time.time()
    text, segment_rows = video_analysis.transcribe_audio(
        AUDIO_PATH, model_size=size, device=TRANSCRIBE_DEVICE, compute_type=TRANSCRIBE_COMPUTE_TYPE,
    )
    latency = time.time() - t0
    transcripts[candidate_name] = text
    transcript_segments[candidate_name] = segment_rows
    log_result(
        'transcript', candidate_name, quality_score=None, cost_usd=0.0, latency_sec=latency,
        notes=f'{len(text)} karakter, {len(segment_rows)} segment, VAD+timestamps',
    )
    gc.collect()

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    # gpt-4o-mini-transcribe: whisper-1'in yerini alan guncel, daha ucuz model.
    with open(AUDIO_PATH, 'rb') as f:
        resp = client.audio.transcriptions.create(model='gpt-4o-mini-transcribe', file=f)
    latency = time.time() - t0
    transcripts['openai-gpt-4o-mini-transcribe'] = resp.text
    cost = (CLIP_DURATION_SEC / 60) * 0.003  # yaklasik; platform.openai.com/docs/pricing
    log_result('transcript', 'openai-gpt-4o-mini-transcribe', quality_score=None, cost_usd=cost,
               latency_sec=latency, notes=f'{len(resp.text)} karakter')
else:
    print('OpenAI key yok, openai-gpt-4o-mini-transcribe atlandi.')


In [ ]:
# Ciktilari yan yana oku ve kalite puanini (1-5) kendin gir.
for name, text in transcripts.items():
    print('=' * 20, name, '=' * 20)
    print(text[:800], '...\n')

transcript_quality = {
    # ornek: 'faster-whisper-tiny': 2, 'faster-whisper-small': 3, 'faster-whisper-large-v3-turbo': 4
}

for r in results:
    if r['task'] == 'transcript' and r['candidate'] in transcript_quality:
        r['quality_score_1_5'] = transcript_quality[r['candidate']]
print('Kalite puanlarini transcript_quality sozlugune girip bu hucreyi tekrar calistir.')

## 3. Sahne Tespiti — Aday Yöntemler

Adaylar: PySceneDetect `ContentDetector` ve `AdaptiveDetector`. Kalite ölçütü: bulunan sahne sayısının videoyu mantıklı şekilde bölüp bölmediği (elle kontrol).

In [ ]:
scene_results = {}

# Kanonik detect_scenes() iki dedektorle karsilastirmali calistirilir.
for name, detector_key in [('content-detector', 'content'), ('adaptive-detector', 'adaptive')]:
    t0 = time.time()
    scene_list = video_analysis.detect_scenes(VIDEO_PATH, detector=detector_key)
    latency = time.time() - t0
    scene_results[name] = scene_list
    log_result('scene-detection', name, quality_score=None, cost_usd=0.0, latency_sec=latency,
               notes=f'{len(scene_list)} sahne bulundu')


In [ ]:
scene_quality = {
    # ornek: 'content-detector': 4, 'adaptive-detector': 3
}
for r in results:
    if r['task'] == 'scene-detection' and r['candidate'] in scene_quality:
        r['quality_score_1_5'] = scene_quality[r['candidate']]
print('Sahne sayilarini yukarida gordukten sonra scene_quality sozlugunu doldurup bu hucreyi tekrar calistir.')

## 4. Ses Enerjisi Analizi (Destek Sinyali)

Bu adım tek bir sabit yöntemle (librosa RMS enerji tepe noktaları) çalışır — aday model karşılaştırması gerektirmez, sadece hot-moment skorlamasına ek sinyal sağlar. Bu yüzden burada kalite/maliyet karşılaştırması yapılmaz.

In [ ]:
import numpy as np

t0 = time.time()
# Kanonik compute_audio_peaks(): 16kHz yukleme + RMS + min-mesafeli tepe gruplama.
audio_info = video_analysis.compute_audio_peaks(AUDIO_PATH, top_n=10, min_distance_sec=1.5)
times, rms = audio_info['times'], audio_info['rms']
audio_peak_details = audio_info['peaks']
audio_peaks = [item['time_sec'] for item in audio_peak_details]
latency = time.time() - t0
log_result('audio-energy', 'librosa-rms', quality_score='n/a', cost_usd=0.0, latency_sec=latency,
           notes=f'gruplanmis en yuksek enerjili anlar: {[round(t, 1) for t in audio_peaks]}')


## 5. Yerel + Opsiyonel LLM Hot-Moment Skorlama

Önce transcript, sahne ve ses sinyallerinden API gerektirmeyen yerel hot-moment adayları üretilir. Aynı adaylar, API anahtarı varsa LLM sağlayıcılarıyla karşılaştırılır.

In [ ]:
t0 = time.time()
TRANSCRIPT_PREFERENCE = [
    'faster-whisper-large-v3-turbo',
    'faster-whisper-small',
    'faster-whisper-tiny',
]
BEST_TRANSCRIPT_CANDIDATE = next(
    name for name in TRANSCRIPT_PREFERENCE
    if name in transcript_segments and transcripts.get(name, '').strip()
)
print('Kullanilan transcript:', BEST_TRANSCRIPT_CANDIDATE)

LOCAL_WINDOW_SEC = 30.0
LOCAL_STRIDE_SEC = 10.0
TOP_HOT_MOMENTS = 5

analysis_duration = min(float(CLIP_DURATION_SEC), float(times[-1]))
content_scenes = scene_results.get('content-detector', [])
scene_boundaries = [end for _, end in content_scenes[:-1]] if content_scenes else []

# Kanonik skorlama: master notebook da AYNI score_hot_moments() fonksiyonunu cagirir.
hot_moments = video_analysis.score_hot_moments(
    transcript_segments[BEST_TRANSCRIPT_CANDIDATE], times, rms, scene_boundaries,
    analysis_duration, CLIP_START_SEC, top_n=TOP_HOT_MOMENTS,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
)

# Kanonik onizleme klibi uretimi (her ana bir 720p klip).
video_analysis.generate_preview_clips(FFMPEG_BIN, VIDEO_PATH, hot_moments, RESULTS_DIR / 'hot_moment_previews')

# Kanonik manifest: status HER ZAMAN 'review_required' (asla otomatik approved).
hot_moments_manifest = video_analysis.build_hot_moments_manifest(
    hot_moments, transcript_candidate=BEST_TRANSCRIPT_CANDIDATE,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
    weights=video_analysis.DEFAULT_WEIGHTS,
    raw_video_path=RAW_VIDEO, analysis_clip_path=VIDEO_PATH,
    clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
hot_moments_path = RESULTS_DIR / 'hot_moments.json'
with open(hot_moments_path, 'w', encoding='utf-8') as hot_file:
    json.dump(hot_moments_manifest, hot_file, ensure_ascii=False, indent=2)

local_latency = time.time() - t0
log_result(
    'hotmoment-scoring', 'local-multisignal-v2', quality_score=None, cost_usd=0.0,
    latency_sec=local_latency,
    notes=str([(item['start_sec'], item['end_sec'], item['score']) for item in hot_moments]),
    status='review_required',
)
print('Yerel hot-moment dosyasi:', hot_moments_path, '(status: review_required)')
display(pd.DataFrame(hot_moments)[['rank', 'aligned_start_sec', 'aligned_end_sec', 'score', 'reason', 'preview_path', 'transcript']])

# Opsiyonel LLM karsilastirmasinda yerel yontemin sectigi ayni pencereleri kullan.
windows = [item['transcript'] for item in hot_moments if item['transcript']]
print(f'{len(windows)} zaman kodlu pencere olusturuldu.')

SCORING_PROMPT = (
    "Asagidaki video transkript parcasini oku. 0-100 arasi bir 'hot moment' (vurucu, "
    "paylasilabilir, dikkat cekici an) skoru ver ve 1 cumlelik gerekce yaz. "
    "SADECE JSON don: {{\"score\": <int>, \"reason\": \"...\"}}\n\nMetin:\n{chunk}"
)


In [ ]:
def parse_json_score(raw):
    try:
        start = raw.find('{'); end = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except Exception:
        return {'score': None, 'reason': raw[:200]}

llm_outputs = {'local-multisignal-v2': hot_moments}

# --- OpenAI gpt-5.6-luna (gpt-4o-mini'nin yerini alan dusuk maliyetli model) ---
if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.chat.completions.create(
            model='gpt-5.6-luna',
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
            temperature=0,
        )
        scored.append(parse_json_score(resp.choices[0].message.content))
    latency = time.time() - t0
    llm_outputs['openai-gpt-5.6-luna'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.0  # yaklasik input fiyati, guncelini platform.openai.com/docs/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'openai-gpt-5.6-luna', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('OpenAI key yok, gpt-5.6-luna atlandi.')

# --- Anthropic claude-haiku-4-5 ---
if ANTHROPIC_API_KEY:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.messages.create(
            model='claude-haiku-4-5',
            max_tokens=200,
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
        )
        scored.append(parse_json_score(resp.content[0].text))
    latency = time.time() - t0
    llm_outputs['anthropic-claude-haiku-4-5'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.00  # yaklasik input fiyati, guncelini platform.claude.com/docs/en/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'anthropic-claude-haiku-4-5', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('Anthropic key yok, claude-haiku-4-5 atlandi.')

# --- Google: SABIT MODEL YOK. Her cagri kanonik router uzerinden gider -------
# (local_pipeline/google_ai.py: Rota A = gemini-3.6-flash -> gemini-3.5-flash ->
#  gemini-3.5-flash-lite -> gemma-4-31b-it, ilk basarida durur, kota defteri
#  tutulur, hatalar siniflandirilir ve sanitize edilir.)
#
# Ucretsiz katman veri politikasi: buraya SADECE yerel olarak uretilmis transcript
# pencereleri gonderilir; ham video ASLA yuklenmez.
GOOGLE_CREDENTIAL = google_ai.resolve_secret('google_api', get_secret)
print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL))

if GOOGLE_CREDENTIAL.configured:
    _g_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
    _g_manifest = google_ai.build_google_capability_manifest(
        _g_client, credential=GOOGLE_CREDENTIAL)
    _g_ledger = google_ai.QuotaLedger()
    _g_selection = google_ai.select_google_route(
        'hotmoment_review', manifest=_g_manifest, ledger=_g_ledger)
    print(google_ai.format_route_line(_g_selection, GOOGLE_CREDENTIAL))

    t0 = time.time()
    scored, _g_models, _g_failed = [], [], None
    for chunk in windows:
        _g_result = google_ai.run_google_hotmoment_review(
            SCORING_PROMPT.format(chunk=chunk), client=_g_client,
            manifest=_g_manifest, ledger=_g_ledger,
            required_fields=(), checks=(),
        )
        if _g_result['status'] != 'ok':
            _g_failed = _g_result['status_reason']
            break
        _g_models.append(_g_result['model'])
        scored.append(parse_json_score(_g_result['output']))
    latency = time.time() - t0

    if _g_failed:
        # Cikti UYDURULMAZ: basarisizlik acikca 'failed' olarak loglanir.
        log_result('hotmoment-scoring', 'google-router', quality_score=None, cost_usd=0.0,
                   latency_sec=latency, notes=f'Google rotasi basarisiz: {_g_failed}',
                   status='failed')
        print('Google hot-moment incelemesi basarisiz:', _g_failed)
    else:
        _g_label = 'google-' + (sorted(set(_g_models))[0] if _g_models else 'router')
        llm_outputs[_g_label] = scored
        # Ucretsiz katman: dolar maliyeti 0; gercek butce RPM/TPM/RPD kotasidir.
        log_result('hotmoment-scoring', _g_label, quality_score=None, cost_usd=0.0,
                   latency_sec=latency,
                   notes=str([s.get('score') for s in scored])
                         + f' | kullanilan modeller: {sorted(set(_g_models))}')
else:
    print('Google key yok (GeminiKey / GOOGLE_API_KEY), Google adayi atlandi.')
    print(google_ai.credential_help_text('google_api'))

In [ ]:
for name, scored in llm_outputs.items():
    print('=' * 20, name, '=' * 20)
    for s in scored:
        print(s)
    print()

hotmoment_quality = {
    # ornek: 'local-multisignal-v2': 4, 'openai-gpt-5.6-luna': 4
}
for r in results:
    if r['task'] == 'hotmoment-scoring' and r['candidate'] in hotmoment_quality:
        r['quality_score_1_5'] = hotmoment_quality[r['candidate']]
print('Skorlarin tutarliligini/isabetini degerlendirip hotmoment_quality sozlugunu doldur, sonra tekrar calistir.')

## 6. Karşılaştırma Tablosu ve Dışa Aktarım

In [ ]:
df = pd.DataFrame(results)
# Not: satir bazli status artik log_result tarafindan ayarlaniyor; eksikleri guvenli varsayilana tamamla.
if 'status' not in df.columns:
    df['status'] = 'experimental'
else:
    df['status'] = df['status'].fillna('experimental')
display(df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / '01_video_analysis_hotmoment_sonuclar.csv'
df.to_csv(out_path, index=False)
print('Kaydedildi:', out_path)
print('\nBu dosyayi indirip experiments/results/ altina koy; roadmap Faz 1deki provider registry karari bu sonuclara gore verilecek.')

## 7. Tanılama Paketini İndir

Bu hücre transcriptleri, sahne zamanlarını, ses tepe noktalarını ve karşılaştırma CSV'sini tek bir ZIP dosyasında toplar.

In [ ]:
import zipfile

scene_report = {}
for detector_name, scenes in globals().get('scene_results', {}).items():
    scene_report[detector_name] = {
        'count': len(scenes),
        # detect_scenes() zaten (start_sec, end_sec) float ikilileri dondurur.
        'scenes': [{'start_sec': round(start, 3), 'end_sec': round(end, 3)} for start, end in scenes],
    }

diagnostic_report = {
    'status': 'experimental',
    'status_reason': 'provider comparison notebook output; not a final production artifact',
    'source': {
        'raw_video': globals().get('RAW_VIDEO'),
        'video_path': globals().get('VIDEO_PATH'),
        'audio_path': globals().get('AUDIO_PATH'),
        'clip_start_sec': globals().get('CLIP_START_SEC'),
        'clip_duration_sec': globals().get('CLIP_DURATION_SEC'),
    },
    'transcripts': {
        name: {
            'character_count': len(text),
            'full_text': text,
            'segments': globals().get('transcript_segments', {}).get(name, []),
        }
        for name, text in globals().get('transcripts', {}).items()
    },
    'scene_detection': scene_report,
    'audio_peaks_sec': globals().get('audio_peaks', []),
    'audio_peak_details': globals().get('audio_peak_details', []),
    'local_hot_moments': globals().get('hot_moments', []),
    'llm_hotmoment_outputs': globals().get('llm_outputs', {}),
    'benchmark_results': globals().get('results', []),
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = RESULTS_DIR / '01_video_analysis_tanilama.json'
zip_path = RESULTS_DIR / '01_video_analysis_paylasim_paketi.zip'

with open(report_path, 'w', encoding='utf-8') as report_file:
    json.dump(diagnostic_report, report_file, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, arcname=report_path.name)
    if Path(out_path).exists():
        archive.write(out_path, arcname=Path(out_path).name)
    if Path(globals().get('hot_moments_path', '')).exists():
        archive.write(hot_moments_path, arcname=Path(hot_moments_path).name)
    for preview_file in (RESULTS_DIR / 'hot_moment_previews').glob('*.mp4'):
        archive.write(preview_file, arcname=f'hot_moment_previews/{preview_file.name}')

print('Paylasim paketi hazir:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('Otomatik indirme baslatilamadi. Files panelinden elle indir:', zip_path)
    print('Detay:', exc)


## 8. Karar (elle doldurulacak)

Yukarıdaki tabloyu inceledikten sonra bu hücreyi düzenleyip kararını ve kısa gerekçeni yaz. Bu karar Faz 1'de `packages/providers/vision-analysis` ve `packages/providers/llm` adaptörlerinin varsayılan sağlayıcısını belirleyecek.

- **Transcript:** _(seçilen aday)_ — gerekçe: _...
- **Sahne tespiti:** _(seçilen aday)_ — gerekçe: _...
- **Hot-moment skorlama:** _(seçilen aday)_ — gerekçe: _...
